# Predicting Who Leaves Is Not Knowing Why

Validating churn-model explanations against customers' stated reasons. Run the cells in order; see the repository README for data and runtimes.


## Step 01: Preprocessing, leakage audit, reason grouping and EDA


In [ ]:
# =====================================================================
# NB1 (single cell): Leakage-Free Preprocessing, Reason Grouping & EDA
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
# Output: /kaggle/working/telco_clean.csv, config.json, figures/, tables/
# =====================================================================


# ---------------------------------------------------------------------
# NB1 — Leakage-Free Preprocessing, Reason Grouping & EDA
# ---------------------------------------------------------------------
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = os.environ.get("NB_DATA") or next(iter(glob.glob(os.path.join(DATA_DIR, "**", "Telco_customer_churn.xlsx"), recursive=True)), "Telco_customer_churn.xlsx")
OUT_DIR   = os.environ.get("NB_OUT", WORK_DIR)
FIG_DIR   = os.path.join(OUT_DIR, "figures")
TAB_DIR   = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)
SEED = 42

# Paper-ready plot style: churn = warm, retained = neutral blue
C_CHURN, C_STAY, C_ACCENT = "#C8553D", "#4A6FA5", "#2F2F2F"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11,
})

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

def savetab(df, name, index=True):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=index)
    return df

# ---------------------------------------------------------------------
# 1. Load raw data
# ---------------------------------------------------------------------
raw = pd.read_excel(DATA_PATH, sheet_name="Telco_Churn")
print("Raw shape:", raw.shape)
display(raw.head(3))

# ---------------------------------------------------------------------
# 2. Column roles
# ---------------------------------------------------------------------
ID_COL, TARGET, VALUE_COL = "CustomerID", "Churn Value", "CLTV"
CONSTANT_COLS  = [c for c in raw.columns if raw[c].nunique(dropna=False) == 1]
REDUNDANT_COLS = ["Lat Long", "City", "Zip Code"]
LEAKAGE_COLS   = ["Churn Label", "Churn Score", "Churn Reason"]
print("Constant :", CONSTANT_COLS)
print("Redundant:", REDUNDANT_COLS)
print("Leakage  :", LEAKAGE_COLS)

# ---------------------------------------------------------------------
# 3. Leakage audit (paper er jonno Table)
# ---------------------------------------------------------------------
y_raw = raw[TARGET].astype(int)
audit_rows = [
    ("Churn Label",               roc_auc_score(y_raw, (raw["Churn Label"] == "Yes").astype(int)), "Target er copy"),
    ("Churn Score",               roc_auc_score(y_raw, raw["Churn Score"]),                        "IBM SPSS model er predicted score"),
    ("Churn Reason (non-null)",   roc_auc_score(y_raw, raw["Churn Reason"].notna().astype(int)),   "Shudhu churner der e reason ache"),
    ("CLTV",                      roc_auc_score(y_raw, raw["CLTV"]),                               "Business value — feature hishebe deya hobe na"),
]
leak = pd.DataFrame(audit_rows, columns=["Column", "Standalone AUC", "Interpretation"]).set_index("Column")
leak["Standalone AUC"] = leak["Standalone AUC"].round(4)
print(pd.crosstab(raw["Churn Reason"].notna(), raw["Churn Label"], rownames=["Reason present"]))
display(savetab(leak, "T_leakage_audit"))

# ---------------------------------------------------------------------
# 4. Cleaning
# ---------------------------------------------------------------------
df = raw.copy()

# 4.1 Total Charges: text -> numeric
tc = pd.to_numeric(df["Total Charges"].astype(str).str.strip().replace({"": np.nan, "nan": np.nan}), errors="coerce")
blank = tc.isna()
print("Blank Total Charges:", int(blank.sum()))
print(df.loc[blank, ["Tenure Months", "Monthly Charges", TARGET]].value_counts().head(15))
print("Customers with Tenure = 0 (overall):", int((df["Tenure Months"] == 0).sum()))

if (df.loc[blank, "Tenure Months"] == 0).all():
    tc[blank] = 0.0                       # new customers, not billed yet
    print("-> All blanks have tenure 0: filled with 0 (not yet billed).")
else:
    tc[blank] = df.loc[blank, "Monthly Charges"] * df.loc[blank, "Tenure Months"]
    print("-> Some blanks have tenure > 0: filled with Monthly x Tenure. CHECK MANUALLY.")
df["Total Charges"] = tc.astype(float)

# 4.2 Yes/No -> 1/0 ; Gender -> Male (1/0)
BIN_FEATURES = [c for c in df.columns
                if c not in LEAKAGE_COLS + [TARGET]
                and set(df[c].dropna().unique()) <= {"Yes", "No"}]
for c in BIN_FEATURES:
    df[c] = (df[c] == "Yes").astype(int)
df["Male"] = (df["Gender"] == "Male").astype(int)
BIN_FEATURES = BIN_FEATURES + ["Male"]

# 4.3 Multi-category features stay as strings (NB2 pipeline e encode hobe, fold er bhitore)
CAT_FEATURES = ["Multiple Lines", "Internet Service", "Online Security", "Online Backup",
                "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies",
                "Contract", "Payment Method"]
for c in CAT_FEATURES:
    df[c] = df[c].astype(str).str.strip()

print("Binary:", BIN_FEATURES)
print("Categorical:", CAT_FEATURES)

# ---------------------------------------------------------------------
# 4.4 Light feature engineering (row-wise, tai leakage nai)
# ---------------------------------------------------------------------
SERVICE_COLS = ["Online Security", "Online Backup", "Device Protection", "Tech Support",
                "Streaming TV", "Streaming Movies"]
df["Num Services"] = (df[SERVICE_COLS] == "Yes").sum(axis=1) + df["Phone Service"] + (df["Multiple Lines"] == "Yes").astype(int)
df["Avg Monthly Spend"] = np.where(df["Tenure Months"] > 0,
                                   df["Total Charges"] / df["Tenure Months"].replace(0, np.nan),
                                   df["Monthly Charges"])
df["Charge Gap"] = df["Monthly Charges"] - df["Avg Monthly Spend"]

NUM_FEATURES = ["Tenure Months", "Monthly Charges", "Total Charges", "Latitude", "Longitude",
                "Num Services", "Avg Monthly Spend", "Charge Gap"]
display(df[NUM_FEATURES].describe().T.round(2))

# ---------------------------------------------------------------------
# 5. Churn Reason → Reason Group (validation ground truth)
# ---------------------------------------------------------------------
REASON_GROUPS = {
    "Competitor": ["Competitor offered higher download speeds", "Competitor offered more data",
                   "Competitor made better offer", "Competitor had better devices"],
    "Price":      ["Price too high", "Extra data charges", "Long distance charges",
                   "Lack of affordable download/upload speed"],
    "Support":    ["Attitude of support person", "Attitude of service provider",
                   "Poor expertise of phone support", "Poor expertise of online support"],
    "Quality":    ["Network reliability", "Product dissatisfaction", "Service dissatisfaction",
                   "Lack of self-service on Website", "Limited range of services"],
    "Uncontrollable": ["Moved", "Deceased"],
    "Unknown":    ["Don't know"],
}
reason_to_group = {r: g for g, rs in REASON_GROUPS.items() for r in rs}

df["Churn Reason"] = df["Churn Reason"].astype("string").str.strip().str.replace("\u2019", "'", regex=False)
df["Reason Group"] = df["Churn Reason"].map(reason_to_group)

unmapped = df.loc[df["Churn Reason"].notna() & df["Reason Group"].isna(), "Churn Reason"].unique()
assert len(unmapped) == 0, f"Unmapped reasons: {unmapped}"
assert df.loc[df[TARGET] == 1, "Reason Group"].notna().all(), "Some churners have no reason"

df["Controllable"] = np.where(df[TARGET] == 0, np.nan,
                              (~df["Reason Group"].isin(["Uncontrollable"])).astype(float))

rg = (df.loc[df[TARGET] == 1, "Reason Group"].value_counts()
        .rename("Count").to_frame())
rg["% of churners"] = (rg["Count"] / rg["Count"].sum() * 100).round(1)
rg["CLTV lost (sum)"] = df.loc[df[TARGET] == 1].groupby("Reason Group")[VALUE_COL].sum()
rg["Mean CLTV"] = df.loc[df[TARGET] == 1].groupby("Reason Group")[VALUE_COL].mean().round(0)
display(savetab(rg, "T_reason_groups"))

# ---------------------------------------------------------------------
# 6. Final feature set + save
# ---------------------------------------------------------------------
FEATURES = NUM_FEATURES + BIN_FEATURES + CAT_FEATURES
assert not set(FEATURES) & set(LEAKAGE_COLS + [VALUE_COL, TARGET]), "Leakage column in features!"

KEEP = [ID_COL] + FEATURES + [TARGET, VALUE_COL, "Churn Reason", "Reason Group", "Controllable"]
clean = df[KEEP].copy()
clean.to_csv(os.path.join(OUT_DIR, "telco_clean.csv"), index=False)

config = {
    "id_col": ID_COL, "target": TARGET, "value_col": VALUE_COL,
    "num_features": NUM_FEATURES, "bin_features": BIN_FEATURES, "cat_features": CAT_FEATURES,
    "features": FEATURES, "leakage_cols": LEAKAGE_COLS,
    "dropped_cols": CONSTANT_COLS + REDUNDANT_COLS + ["Gender"],
    "reason_groups": REASON_GROUPS, "seed": SEED,
}
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Saved telco_clean.csv:", clean.shape)
print("n features:", len(FEATURES), "| churn rate:", round(clean[TARGET].mean(), 4))

# ---------------------------------------------------------------------
# 7. Table 1 — Descriptive statistics by churn status
# ---------------------------------------------------------------------
rows = []
for c in NUM_FEATURES + [VALUE_COL]:
    s0, s1 = df.loc[df[TARGET] == 0, c], df.loc[df[TARGET] == 1, c]
    rows.append((c, f"{s0.mean():.2f} ± {s0.std():.2f}", f"{s1.mean():.2f} ± {s1.std():.2f}"))
for c in BIN_FEATURES:
    rows.append((c + " (%)", f"{df.loc[df[TARGET]==0, c].mean()*100:.1f}", f"{df.loc[df[TARGET]==1, c].mean()*100:.1f}"))
t1 = pd.DataFrame(rows, columns=["Variable", f"Retained (n={int((df[TARGET]==0).sum())})",
                                 f"Churned (n={int((df[TARGET]==1).sum())})"]).set_index("Variable")
display(savetab(t1, "T1_descriptives"))

# ---------------------------------------------------------------------
# 8. EDA figures
# ---------------------------------------------------------------------
# Fig 1: churn rate by key categorical drivers
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
overall = df[TARGET].mean() * 100
for ax, col in zip(axes.ravel(), ["Contract", "Internet Service", "Payment Method", "Tech Support"]):
    rate = df.groupby(col)[TARGET].mean().mul(100).sort_values()
    ax.barh(rate.index, rate.values, color=C_CHURN)
    ax.axvline(overall, ls="--", lw=1, color=C_ACCENT)
    for i, v in enumerate(rate.values):
        ax.text(v + 0.8, i, f"{v:.1f}%", va="center", fontsize=9)
    ax.set_title(col); ax.set_xlabel("Churn rate (%)"); ax.set_xlim(0, rate.max() * 1.2)
fig.suptitle(f"Churn rate by customer attribute (dashed = overall {overall:.1f}%)", y=1.01)
savefig("F1_churn_rate_by_category")
# Fig 2: tenure, monthly charges, charge gap distributions
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col in zip(axes, ["Tenure Months", "Monthly Charges", "Charge Gap"]):
    for val, colr, lab in [(0, C_STAY, "Retained"), (1, C_CHURN, "Churned")]:
        sns.kdeplot(df.loc[df[TARGET] == val, col], ax=ax, color=colr, fill=True, alpha=0.25, lw=1.5, label=lab, common_norm=False)
    ax.set_title(col); ax.set_ylabel("Density")
axes[0].legend(frameon=False)
savefig("F2_numeric_distributions")
# Fig 3: reason groups (count) + 20 detailed reasons
churners = df[df[TARGET] == 1]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.6]})
g = churners["Reason Group"].value_counts().sort_values()
axes[0].barh(g.index, g.values, color=C_CHURN)
for i, v in enumerate(g.values):
    axes[0].text(v + 5, i, f"{v} ({v/len(churners)*100:.0f}%)", va="center", fontsize=9)
axes[0].set_title("Churn reason groups"); axes[0].set_xlim(0, g.max() * 1.3)

r = churners["Churn Reason"].value_counts().sort_values()
axes[1].barh(r.index, r.values, color=[C_STAY] * len(r))
axes[1].set_title("Stated churn reasons (n = %d churners)" % len(churners))
savefig("F3_churn_reasons")
# Fig 4: reason group x contract / internet service (row %): which customers leave for which reason
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["Contract", "Internet Service"]):
    ct = pd.crosstab(churners["Reason Group"], churners[col], normalize="index").mul(100)
    sns.heatmap(ct, annot=True, fmt=".0f", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"Reason group × {col} (row %)"); ax.set_ylabel("")
savefig("F4_reason_by_profile")
# Fig 5: geographic churn map (zip-level, >= 5 customers)
geo = (raw.assign(churn=raw[TARGET])
          .groupby("Zip Code").agg(lat=("Latitude", "mean"), lon=("Longitude", "mean"),
                                   n=("churn", "size"), rate=("churn", "mean")))
geo = geo[geo["n"] >= 5]
fig, ax = plt.subplots(figsize=(6.5, 7.5))
sc = ax.scatter(geo["lon"], geo["lat"], c=geo["rate"] * 100, s=geo["n"] * 2,
                cmap="RdYlBu_r", alpha=0.8, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Churn rate (%)", shrink=0.7)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Zip-level churn rate, California ({len(geo)} zips with ≥5 customers)")
savefig("F5_geo_churn_map")
# Fig 6: CLTV at risk by reason group
fig, ax = plt.subplots(figsize=(8, 4))
order = churners.groupby("Reason Group")[VALUE_COL].median().sort_values().index
sns.boxplot(data=churners, y="Reason Group", x=VALUE_COL, order=order, color=C_STAY, fliersize=2, ax=ax)
ax.axvline(df.loc[df[TARGET] == 0, VALUE_COL].median(), ls="--", color=C_ACCENT, lw=1)
ax.set_title("CLTV of churned customers by reason group (dashed = retained median)"); ax.set_ylabel("")
savefig("F6_cltv_by_reason")

# ---------------------------------------------------------------------
# 9. Draft: feature → reason-domain map (NB3 er jonno)
# ---------------------------------------------------------------------
FEATURE_DOMAIN = {
    "Price":   ["Monthly Charges", "Total Charges", "Avg Monthly Spend", "Charge Gap"],
    "Support": ["Tech Support", "Online Security", "Device Protection"],
    "Quality/Competitor (offering)": ["Internet Service", "Streaming TV", "Streaming Movies",
                                      "Multiple Lines", "Online Backup", "Phone Service", "Num Services"],
    "Relationship (no direct reason)": ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"],
    "Demographic/Geo (no direct reason)": ["Male", "Senior Citizen", "Partner", "Dependents", "Latitude", "Longitude"],
}
mapped = {f for fs in FEATURE_DOMAIN.values() for f in fs}
print("Features not mapped:", set(FEATURES) - mapped)
config["feature_domain_draft"] = FEATURE_DOMAIN
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)
print("Outputs:", sorted(os.listdir(OUT_DIR)))
print("Figures:", sorted(os.listdir(FIG_DIR)))
print("Tables :", sorted(os.listdir(TAB_DIR)))

## Step 02: Stage 1: risk models under nested repeated CV


In [ ]:
# =====================================================================
# NB2 (single cell): Stage-1 Churn Prediction with Nested Repeated CV
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
# Solves: L3 (no nested CV / no stat tests), L4 (SMOTE leakage),
#         L9 (calibration ignored)
# Input : NB1 output (telco_clean.csv, config.json)
# Output: /kaggle/working/nb2/  -> fold metrics, OOF predictions,
#         best params, stat tests, figures (300 dpi)
#
# RUNTIME: see MODE below (balanced ~45-70 min on Kaggle CPU). Use "Save Version ->
#          Save & Run All (Commit)" so it runs in the background.
#          MODE = "fast" first to test everything in ~10 min.
# =====================================================================

# ---------------------------------------------------------------------
# 0. Settings
# ---------------------------------------------------------------------
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

MODE = os.environ.get("MODE", "balanced")  # "fast" (~10 min test) | "balanced" (~45-70 min) | "full" (~2-3 h)

N_SPLITS   = 5                                                   # outer folds
N_REPEATS  = {"fast": 1, "balanced": 2, "full": 3}[MODE]         # 5 x 2 = 10 outer folds
INNER_K    = 3                                                   # inner folds for tuning
N_TRIALS   = {"fast": 5, "balanced": 15, "full": 25}[MODE]       # Optuna trials per outer fold
FAST       = MODE == "fast"
RESUME     = True     # True = already-finished models are loaded from disk, not retrained
SEED       = 42

import os, json, time, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, brier_score_loss, log_loss,
                             roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise ImportError("Optuna nai. Kaggle e sadharonoto thake; na thakle: !pip install optuna")

AVAILABLE = {}
try:
    from xgboost import XGBClassifier; AVAILABLE["XGBoost"] = True
except ImportError: print("xgboost not found -> skipped")
try:
    from lightgbm import LGBMClassifier; AVAILABLE["LightGBM"] = True
except ImportError: print("lightgbm not found -> skipped")
try:
    from catboost import CatBoostClassifier; AVAILABLE["CatBoost"] = True
except ImportError: print("catboost not found -> skipped")
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE; HAS_IMB = True
except ImportError:
    HAS_IMB = False; print("imblearn not found -> SMOTE ablation skipped")

# ---------------------------------------------------------------------
# 1. Paths + load NB1 output
# ---------------------------------------------------------------------
def find_file(name):
    """NB1 output: same notebook (/kaggle/working) or attached as input."""
    for p in [os.path.join(os.environ.get("NB_IN", WORK_DIR), name)] + \
             glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} pai nai. Age NB1 run koro, ba NB1 er output notebook ta input hishebe attach koro.")

OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb2")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)

cfg  = json.load(open(find_file("config.json")))
data = pd.read_csv(find_file("telco_clean.csv"))
TARGET = cfg["target"]

# Drop near-duplicate of Monthly Charges (would split SHAP credit, hurts reason validation)
DROP_FEATS = ["Avg Monthly Spend"]
NUM  = [f for f in cfg["num_features"] if f not in DROP_FEATS]
BIN  = cfg["bin_features"]
CAT  = cfg["cat_features"]
FEATS = NUM + BIN + CAT
assert not set(FEATS) & set(cfg["leakage_cols"] + [cfg["value_col"], TARGET])

X = data[FEATS].copy()
y = data[TARGET].astype(int).values
POS_W = (y == 0).sum() / (y == 1).sum()
print(f"Data: {X.shape} | churn rate {y.mean():.4f} | pos weight {POS_W:.2f}")
print(f"Outer CV: {N_SPLITS}x{N_REPEATS} = {N_SPLITS*N_REPEATS} folds | inner {INNER_K}-fold | {N_TRIALS} trials")

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"{name}.png"), bbox_inches="tight"); plt.show()

# ---------------------------------------------------------------------
# 2. Preprocessing (fit INSIDE each fold via Pipeline -> no leakage)
# ---------------------------------------------------------------------
def make_pre():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("bin", "passthrough", BIN),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
    ])

# ---------------------------------------------------------------------
# 3. Models + Optuna search spaces
#    Imbalance handled by class weights (SMOTE tested separately as ablation)
# ---------------------------------------------------------------------
def build(name, p, seed):
    if name == "LogReg":
        return LogisticRegression(C=p["C"], class_weight="balanced", max_iter=3000)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                                      min_samples_leaf=p["min_samples_leaf"], max_features=p["max_features"],
                                      class_weight="balanced", n_jobs=-1, random_state=seed)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                             learning_rate=p["learning_rate"], subsample=p["subsample"],
                             colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
                             reg_lambda=p["reg_lambda"], scale_pos_weight=POS_W,
                             eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=seed, verbosity=0)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=p["n_estimators"], num_leaves=p["num_leaves"],
                              learning_rate=p["learning_rate"], min_child_samples=p["min_child_samples"],
                              subsample=p["subsample"], subsample_freq=1, colsample_bytree=p["colsample_bytree"],
                              reg_lambda=p["reg_lambda"], class_weight="balanced",
                              n_jobs=-1, random_state=seed, verbose=-1)
    if name == "CatBoost":
        return CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=p["learning_rate"],
                                  l2_leaf_reg=p["l2_leaf_reg"], auto_class_weights="Balanced",
                                  random_seed=seed, verbose=0, thread_count=-1)
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=p["hidden"], alpha=p["alpha"], learning_rate_init=p["lr"],
                             early_stopping=True, max_iter=500, random_state=seed)

MLP_ARCH = {"32": (32,), "64": (64,), "64-32": (64, 32)}

def suggest(name, t):
    if name == "LogReg":
        return {"C": t.suggest_float("C", 1e-3, 10, log=True)}
    if name == "RandomForest":
        return {"n_estimators": t.suggest_int("n_estimators", 200, 600, step=100),
                "max_depth": t.suggest_int("max_depth", 3, 16),
                "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 20),
                "max_features": t.suggest_categorical("max_features", ["sqrt", 0.3, 0.5])}
    if name == "XGBoost":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 600, step=50),
                "max_depth": t.suggest_int("max_depth", 2, 6),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "subsample": t.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": t.suggest_int("min_child_weight", 1, 10),
                "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True)}
    if name == "LightGBM":
        return {"n_estimators": t.suggest_int("n_estimators", 100, 600, step=50),
                "num_leaves": t.suggest_int("num_leaves", 8, 64),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "min_child_samples": t.suggest_int("min_child_samples", 10, 100),
                "subsample": t.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True)}
    if name == "CatBoost":
        return {"iterations": t.suggest_int("iterations", 200, 600, step=100),
                "depth": t.suggest_int("depth", 3, 7),
                "learning_rate": t.suggest_float("learning_rate", 0.02, 0.2, log=True),
                "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1, 10, log=True)}
    if name == "MLP":
        return {"hidden": MLP_ARCH[t.suggest_categorical("hidden", list(MLP_ARCH))],
                "alpha": t.suggest_float("alpha", 1e-5, 1e-1, log=True),
                "lr": t.suggest_float("lr", 1e-4, 1e-2, log=True)}

MODELS = ["LogReg", "RandomForest"] + [m for m in ["XGBoost", "LightGBM", "CatBoost"] if m in AVAILABLE] + ["MLP"]
print("Models:", MODELS)

def pipe(name, p, seed):
    return Pipeline([("pre", make_pre()), ("clf", build(name, p, seed))])

def metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    return {"AUC": roc_auc_score(y_true, prob), "PR_AUC": average_precision_score(y_true, prob),
            "F1": f1_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred), "BalAcc": balanced_accuracy_score(y_true, pred),
            "Brier": brier_score_loss(y_true, prob), "LogLoss": log_loss(y_true, np.clip(prob, 1e-7, 1 - 1e-7))}

# ---------------------------------------------------------------------
# 4. Nested repeated CV (outer = evaluation, inner = Optuna tuning)
#    Same outer splits for every model -> paired statistical tests
# ---------------------------------------------------------------------
outer = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
SPLITS = list(outer.split(X, y))
json.dump([{"fold": i, "repeat": i // N_SPLITS, "test_idx": te.tolist()} for i, (_, te) in enumerate(SPLITS)],
          open(os.path.join(OUT, "outer_splits.json"), "w"))

fold_rows, oof_rows, param_rows = [], [], []
BEST = {}   # (model, fold) -> Optuna best_params

INT_P = {"n_estimators", "max_depth", "min_samples_leaf", "num_leaves", "min_child_samples",
         "min_child_weight", "iterations", "depth"}
def clean_params(d):
    """Restore Optuna params read back from CSV (NaN removal + type fixes)."""
    out = {}
    for kk, v in d.items():
        if kk in ("Model", "Fold") or (isinstance(v, float) and np.isnan(v)):
            continue
        if kk in INT_P: v = int(v)
        elif kk == "max_features": v = v if str(v) == "sqrt" else float(v)
        elif kk == "hidden": v = str(int(v)) if isinstance(v, (int, float)) else str(v)
        else: v = float(v)
        out[kk] = v
    return out

DONE = set()
paths = {n: os.path.join(OUT, f) for n, f in [("fm", "fold_metrics.csv"), ("oof", "oof_predictions.csv"), ("bp", "best_params.csv")]}
if RESUME and all(os.path.exists(p) for p in paths.values()):
    prev_fm, prev_oof, prev_bp = (pd.read_csv(paths[n]) for n in ["fm", "oof", "bp"])
    same_cv = prev_fm["Fold"].nunique() == len(SPLITS) and \
              set(prev_oof.loc[(prev_oof.Model == prev_oof.Model.iloc[0]) & (prev_oof.Fold == 0), "row"]) == set(SPLITS[0][1].tolist())
    if same_cv:
        DONE = {m for m in MODELS if (prev_fm.Model == m).sum() == len(SPLITS) and (prev_bp.Model == m).sum() == len(SPLITS)}
        fold_rows  = prev_fm[prev_fm.Model.isin(DONE)].to_dict("records")
        oof_rows   = [prev_oof[prev_oof.Model.isin(DONE)]]
        param_rows = prev_bp[prev_bp.Model.isin(DONE)].to_dict("records")
        for r in param_rows:
            BEST[(r["Model"], int(r["Fold"]))] = clean_params(r)
        print("RESUME: loaded finished models ->", sorted(DONE))
    else:
        print("RESUME: saved results use a different CV setup -> retraining everything")

t_all = time.time()
for name in MODELS:
    if name in DONE:
        print(f"{name:13s} skipped (loaded from disk)")
        continue
    t0 = time.time()
    for k, (tr, te) in enumerate(SPLITS):
        Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
        inner = StratifiedKFold(n_splits=INNER_K, shuffle=True, random_state=SEED + k)

        def objective(trial):
            p = suggest(name, trial)
            cv_jobs = INNER_K if name in ["LogReg", "MLP"] else 1   # LogReg/MLP single-threaded -> run folds in parallel
            return cross_val_score(pipe(name, p, SEED), Xtr, ytr, cv=inner, scoring="roc_auc", n_jobs=cv_jobs).mean()

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED + k))
        study.optimize(objective, n_trials=1 if name == "LogReg" and FAST else N_TRIALS, show_progress_bar=False)
        best = suggest(name, optuna.trial.FixedTrial(study.best_params))

        model = pipe(name, best, SEED + k).fit(Xtr, ytr)
        prob = model.predict_proba(Xte)[:, 1]

        print(f"  {name} fold {k+1}/{len(SPLITS)}: AUC {roc_auc_score(yte, prob):.4f} "
              f"| elapsed {(time.time()-t0)/60:.1f} min", flush=True)
        m = metrics(yte, prob); m.update({"Model": name, "Fold": k, "Repeat": k // N_SPLITS, "InnerAUC": study.best_value})
        fold_rows.append(m)
        param_rows.append({"Model": name, "Fold": k, **study.best_params})
        BEST[(name, k)] = study.best_params
        oof_rows.append(pd.DataFrame({"Model": name, "Fold": k, "Repeat": k // N_SPLITS,
                                      "row": te, "y": yte, "prob": prob}))
    folds_df = pd.DataFrame(fold_rows)
    print(f"{name:13s} AUC {folds_df.loc[folds_df.Model==name,'AUC'].mean():.4f} "
          f"± {folds_df.loc[folds_df.Model==name,'AUC'].std():.4f} | {(time.time()-t0)/60:.1f} min")
    # save after each model (Kaggle session crash hole progress thakbe)
    folds_df.to_csv(os.path.join(OUT, "fold_metrics.csv"), index=False)
    pd.concat(oof_rows).to_csv(os.path.join(OUT, "oof_predictions.csv"), index=False)
    pd.DataFrame(param_rows).to_csv(os.path.join(OUT, "best_params.csv"), index=False)
print(f"Total: {(time.time()-t_all)/60:.1f} min")

folds_df = pd.DataFrame(fold_rows)
oof = pd.concat(oof_rows, ignore_index=True)

# ---------------------------------------------------------------------
# 5. Table: performance (mean ± sd over outer folds)
# ---------------------------------------------------------------------
MET = ["AUC", "PR_AUC", "F1", "Precision", "Recall", "BalAcc", "Brier", "LogLoss"]
agg = folds_df.groupby("Model")[MET].agg(["mean", "std"])
perf = pd.DataFrame({m: agg[(m, "mean")].map("{:.4f}".format) + " ± " + agg[(m, "std")].map("{:.4f}".format) for m in MET})
perf = perf.loc[agg[("AUC", "mean")].sort_values(ascending=False).index]
perf.to_csv(os.path.join(TAB, "T2_performance.csv"))
print("\n=== Table 2: Outer-fold performance (mean ± sd) ==="); display(perf)

# ---------------------------------------------------------------------
# 6. Statistical tests
#    (a) Friedman on per-fold AUC ranks + Nemenyi critical difference
#    (b) Nadeau-Bengio corrected resampled t-test vs best model, Holm-adjusted
# ---------------------------------------------------------------------
auc_mat = folds_df.pivot(index="Fold", columns="Model", values="AUC")[list(perf.index)]
k_mod, n_blk = auc_mat.shape[1], auc_mat.shape[0]
fr_stat, fr_p = (stats.friedmanchisquare(*[auc_mat[c].values for c in auc_mat.columns])
                 if k_mod >= 3 else (np.nan, np.nan))
ranks = auc_mat.rank(axis=1, ascending=False).mean().sort_values()
Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}   # Demsar (2006)
CD = Q05[k_mod] * np.sqrt(k_mod * (k_mod + 1) / (6 * n_blk))
print(f"\nFriedman chi2 = {fr_stat:.3f}, p = {fr_p:.2e} | Nemenyi CD (alpha=.05) = {CD:.3f}")

n_te = len(SPLITS[0][1]); n_tr = len(SPLITS[0][0]); J = n_blk
best_model = perf.index[0]
rows = []
for mdl in auc_mat.columns:
    if mdl == best_model: continue
    d = auc_mat[best_model].values - auc_mat[mdl].values
    var_c = (1 / J + n_te / n_tr) * d.var(ddof=1)
    t = d.mean() / np.sqrt(var_c) if var_c > 0 else np.inf
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    rows.append({"Comparison": f"{best_model} vs {mdl}", "Mean ΔAUC": d.mean(), "t": t, "p_raw": p})
tt = pd.DataFrame(rows).sort_values("p_raw").reset_index(drop=True)
m_ = len(tt)
tt["p_holm"] = np.maximum.accumulate(np.minimum(1, tt["p_raw"] * (m_ - np.arange(m_))))
tt["Significant (0.05)"] = tt["p_holm"] < 0.05
stat_tab = pd.concat([ranks.rename("Mean rank").to_frame()], axis=1)
stat_tab.to_csv(os.path.join(TAB, "T3_mean_ranks.csv"))
tt.round(5).to_csv(os.path.join(TAB, "T3_corrected_ttest.csv"), index=False)
print("Mean ranks:"); display(ranks.round(3).to_frame("Mean rank"))
print("Corrected resampled t-test (Holm):"); display(tt.round(5))

# ---------------------------------------------------------------------
# 7. SMOTE ablation (SMOTE strictly INSIDE training fold) — L4
#    Uses first repeat's folds and each fold's tuned params; compares
#    class-weight vs SMOTE for LogReg and best boosting model
# ---------------------------------------------------------------------
if HAS_IMB:
    abl = []
    boost = [m for m in perf.index if m in ["XGBoost", "LightGBM", "CatBoost"]]
    abl_models = ["LogReg"] + boost[:1]
    for name in abl_models:
        for k in range(N_SPLITS):
            tr, te = SPLITS[k]
            p = suggest(name, optuna.trial.FixedTrial(BEST[(name, k)]))
            clf = build(name, p, SEED + k)
            # turn off class weighting for the SMOTE variant
            if name == "LogReg": clf.set_params(class_weight=None)
            elif name == "XGBoost": clf.set_params(scale_pos_weight=1)
            elif name == "LightGBM": clf.set_params(class_weight=None)
            elif name == "CatBoost":   # auto_class_weights=None is not accepted -> rebuild without it
                clf = CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=p["learning_rate"],
                                         l2_leaf_reg=p["l2_leaf_reg"], random_seed=SEED + k, verbose=0, thread_count=-1)
            sm = ImbPipeline([("pre", make_pre()), ("smote", SMOTE(random_state=SEED + k)), ("clf", clf)])
            prob = sm.fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
            r = metrics(y[te], prob); r.update({"Model": name, "Imbalance": "SMOTE (in-fold)", "Fold": k}); abl.append(r)
            r2 = folds_df[(folds_df.Model == name) & (folds_df.Fold == k)][MET].iloc[0].to_dict()
            r2.update({"Model": name, "Imbalance": "Class weight", "Fold": k}); abl.append(r2)
    abl = pd.DataFrame(abl).groupby(["Model", "Imbalance"])[["AUC", "PR_AUC", "F1", "Recall", "Brier"]].mean().round(4)
    abl.to_csv(os.path.join(TAB, "T4_smote_ablation.csv"))
    print("\n=== SMOTE (in-fold) vs class weights, repeat 1 ==="); display(abl)

# ---------------------------------------------------------------------
# 8. Figures
# ---------------------------------------------------------------------
order = list(perf.index)
cols = plt.cm.tab10(np.linspace(0, 1, 10))[:len(order)]
CMAP = dict(zip(order, cols))

# F7: AUC distribution over outer folds
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([auc_mat[m] for m in order], vert=False, patch_artist=True,
           boxprops=dict(facecolor="#DCE6F2"), medianprops=dict(color="#C8553D"))
ax.set_yticks(range(1, len(order) + 1)); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel("AUC (outer folds)")
ax.set_title(f"AUC across {n_blk} outer folds ({N_SPLITS}x{N_REPEATS} nested CV)")
savefig("F7_auc_boxplot")

# F8: Critical difference style plot
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.hlines(0, 1, k_mod, color="k", lw=1)
for i, (mdl, r) in enumerate(ranks.items()):
    ax.plot(r, 0, "o", color=CMAP[mdl]); ax.annotate(f"{mdl} ({r:.2f})", (r, 0), xytext=(r, 0.35 + 0.25 * (i % 3)),
                                                    ha="center", fontsize=9, arrowprops=dict(arrowstyle="-", lw=0.6))
ax.plot([1, 1 + CD], [-0.4, -0.4], color="#C8553D", lw=3); ax.text(1 + CD / 2, -0.7, f"CD = {CD:.2f}", ha="center", fontsize=9)
ax.set_xlim(0.8, k_mod + 0.2); ax.set_ylim(-1, 1.3); ax.set_yticks([]); ax.set_xlabel("Mean rank (lower = better)")
ax.spines["left"].set_visible(False)
ax.set_title(f"Friedman p = {fr_p:.1e}; models closer than CD are not significantly different")
savefig("F8_critical_difference")

# F9: ROC + PR curves (repeat 0 OOF = every customer predicted once)
o0 = oof[oof.Repeat == 0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for mdl in order:
    s = o0[o0.Model == mdl]
    fpr, tpr, _ = roc_curve(s.y, s.prob); pr, rc, _ = precision_recall_curve(s.y, s.prob)
    axes[0].plot(fpr, tpr, color=CMAP[mdl], lw=1.4, label=f"{mdl} ({roc_auc_score(s.y, s.prob):.3f})")
    axes[1].plot(rc, pr, color=CMAP[mdl], lw=1.4, label=f"{mdl} ({average_precision_score(s.y, s.prob):.3f})")
axes[0].plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8)
axes[1].axhline(y.mean(), ls="--", color="grey", lw=0.8)
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC (out-of-fold, repeat 1)")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall (out-of-fold, repeat 1)")
for a in axes: a.legend(frameon=False, fontsize=8)
savefig("F9_roc_pr")

# F10: Calibration (important for profit layer)
fig, ax = plt.subplots(figsize=(5.5, 5))
for mdl in order:
    s = o0[o0.Model == mdl]
    fp, mp = calibration_curve(s.y, s.prob, n_bins=10, strategy="quantile")
    ax.plot(mp, fp, "o-", ms=3, lw=1.2, color=CMAP[mdl], label=mdl)
ax.plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8)
ax.set(xlabel="Mean predicted probability", ylabel="Observed churn rate", title="Calibration (out-of-fold)")
ax.legend(frameon=False, fontsize=8)
savefig("F10_calibration")

print("\nNote: class-weighted models are often over-confident upward (curve below diagonal).")
print("NB5 (profit) e isotonic calibration apply kora hobe inner-fold e.")
print("\nSaved to", OUT, "->", sorted(os.listdir(OUT)))

## Step 03: Stages 2–3: explanations, stability and validation against stated reasons


In [ ]:
# =====================================================================
# NB3 (single cell): Explainability + Reason Validation  (MAIN NOVELTY)
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# RQ2: Do SHAP / LIME explanations agree with customers' own stated
#      churn reasons?
# Solves: L2 (explanations never validated against ground truth),
#         L5 (no stability analysis), L8 (subjective evaluation)
#
# Input : NB1 + NB2 output in /kaggle/working (same notebook)
# Output: /kaggle/working/nb3/ -> SHAP values, tables, figures
# RUNTIME: ~5-15 min (LIME part ~5-10 min; set RUN_LIME=False to skip)
# =====================================================================

import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

RUN_LIME     = True
LIME_SAMPLES = 1000      # perturbations per explained customer
N_PERM       = 5000      # permutation-test iterations
N_BOOT       = 1000      # bootstrap iterations for CIs
SEED         = 42

import os, json, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, cohen_kappa_score, confusion_matrix
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

HAS = {}
try:
    import xgboost as xgb; from xgboost import XGBClassifier; HAS["XGBoost"] = True
except ImportError: print("xgboost not found")
try:
    from catboost import CatBoostClassifier, Pool; HAS["CatBoost"] = True
except ImportError: print("catboost not found")
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    if RUN_LIME: print("lime not found -> run `!pip install lime` in a cell above, or LIME will be skipped")
    RUN_LIME = False

# ---------------------------------------------------------------------
# 1. Load NB1 + NB2 outputs
# ---------------------------------------------------------------------
BASE = os.environ.get("NB_IN", WORK_DIR)
def find_file(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found. Run NB1 and NB2 first.")

OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb3")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)

cfg    = json.load(open(find_file("config.json")))
data   = pd.read_csv(find_file("telco_clean.csv"))
splits = json.load(open(find_file("nb2/outer_splits.json")))
bparam = pd.read_csv(find_file("nb2/best_params.csv"))
TARGET = cfg["target"]

NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]   # same as NB2
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT
X, y = data[FEATS].copy(), data[TARGET].astype(int).values

# Repeat 0 folds: every customer is in exactly one test fold
FOLDS = [s for s in splits if s["repeat"] == 0]
assert sorted(np.concatenate([f["test_idx"] for f in FOLDS])) == list(range(len(data)))

MODELS = [m for m in ["CatBoost", "XGBoost", "LogReg"] if m == "LogReg" or m in HAS]
MODELS = [m for m in MODELS if (bparam.Model == m).any()]
PRIMARY = MODELS[0]
print("Models explained:", MODELS, "| primary:", PRIMARY)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"{name}.png"), bbox_inches="tight"); plt.show()

# ---------------------------------------------------------------------
# 2. Feature -> explanation domain, reason group -> domain
#    PRIMARY mapping + ALTERNATIVE mapping (sensitivity analysis)
# ---------------------------------------------------------------------
DOMAIN_MAP = {
    "Price":    ["Monthly Charges", "Total Charges", "Charge Gap"],
    "Support":  ["Tech Support", "Online Security", "Device Protection"],
    "Offering": ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines",
                 "Online Backup", "Phone Service", "Num Services"],
    "Relationship": ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"],
    "Demographic":  ["Male", "Senior Citizen", "Partner", "Dependents", "Latitude", "Longitude"],
}
# Alternative: security/protection are product add-ons, not support; fibre speed is price-tier related
DOMAIN_MAP_ALT = {
    "Price":    ["Monthly Charges", "Total Charges", "Charge Gap", "Internet Service"],
    "Support":  ["Tech Support"],
    "Offering": ["Online Security", "Device Protection", "Streaming TV", "Streaming Movies",
                 "Multiple Lines", "Online Backup", "Phone Service", "Num Services"],
    "Relationship": DOMAIN_MAP["Relationship"],
    "Demographic":  DOMAIN_MAP["Demographic"],
}
for mp in [DOMAIN_MAP, DOMAIN_MAP_ALT]:
    assert sorted(f for v in mp.values() for f in v) == sorted(FEATS), "Every feature must be mapped once"

REASON_TO_DOMAIN = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
ACTIONABLE = ["Price", "Support", "Offering"]          # domains a stated reason can point to
ALL_DOMAINS = list(DOMAIN_MAP)

# ---------------------------------------------------------------------
# 3. Refit fold models (NB2 tuned params) and compute local explanations
#    SHAP: native TreeSHAP for CatBoost/XGBoost, exact linear SHAP for LogReg
#    (all in log-odds space). One-hot SHAP summed back to original feature.
# ---------------------------------------------------------------------
INT_P = {"n_estimators", "max_depth", "min_samples_leaf", "num_leaves", "min_child_samples",
         "min_child_weight", "iterations", "depth"}
def get_params(model, fold):
    r = bparam[(bparam.Model == model) & (bparam.Fold == fold)].iloc[0].to_dict()
    return {k: (int(v) if k in INT_P else v) for k, v in r.items()
            if k not in ("Model", "Fold") and not (isinstance(v, float) and np.isnan(v))}

def make_pre():
    return ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])

def build(model, p, seed, pos_w):
    if model == "LogReg":
        return LogisticRegression(C=float(p["C"]), class_weight="balanced", max_iter=3000)
    if model == "XGBoost":
        return XGBClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                             learning_rate=float(p["learning_rate"]), subsample=float(p["subsample"]),
                             colsample_bytree=float(p["colsample_bytree"]), min_child_weight=p["min_child_weight"],
                             reg_lambda=float(p["reg_lambda"]), scale_pos_weight=pos_w, eval_metric="logloss",
                             tree_method="hist", n_jobs=-1, random_state=seed, verbosity=0)
    if model == "CatBoost":
        return CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=float(p["learning_rate"]),
                                  l2_leaf_reg=float(p["l2_leaf_reg"]), auto_class_weights="Balanced",
                                  random_seed=seed, verbose=0, thread_count=-1)

def output_to_feature(pre):
    """Original feature name for every transformed column."""
    owners = []
    for name, trans, cols in pre.transformers_:
        if name == "cat":
            for c, cats in zip(cols, trans.categories_): owners += [c] * len(cats)
        elif name in ("num", "bin"):
            owners += list(cols)
    return np.array(owners)

def local_shap(model, clf, Xt_test, Xt_train):
    if model == "CatBoost":
        sv = clf.get_feature_importance(Pool(Xt_test), type="ShapValues")[:, :-1]
    elif model == "XGBoost":
        sv = clf.get_booster().predict(xgb.DMatrix(Xt_test), pred_contribs=True)[:, :-1]
    else:  # linear SHAP (interventional, feature independence): coef * (x - E[x])
        sv = clf.coef_[0] * (Xt_test - Xt_train.mean(axis=0))
    return sv

SHAP = {m: np.zeros((len(data), len(FEATS))) for m in MODELS}
PROB = {m: np.zeros(len(data)) for m in MODELS}
FITTED = {}   # (model, fold) -> pipeline (reused by LIME and NB5)
t0 = time.time()
for f in FOLDS:
    k, te = f["fold"], np.array(f["test_idx"])
    tr = np.setdiff1d(np.arange(len(data)), te)
    pos_w = (y[tr] == 0).sum() / (y[tr] == 1).sum()
    for m in MODELS:
        pipe = Pipeline([("pre", make_pre()), ("clf", build(m, get_params(m, k), SEED + k, pos_w))])
        pipe.fit(X.iloc[tr], y[tr])
        pre, clf = pipe.named_steps["pre"], pipe.named_steps["clf"]
        Xt_te, Xt_tr = pre.transform(X.iloc[te]), pre.transform(X.iloc[tr])
        sv = local_shap(m, clf, Xt_te, Xt_tr)
        owners = output_to_feature(pre)
        agg = pd.DataFrame(sv, columns=owners).T.groupby(level=0).sum().T[FEATS].values   # sum one-hot back
        SHAP[m][te] = agg
        PROB[m][te] = pipe.predict_proba(X.iloc[te])[:, 1]
        FITTED[(m, k)] = pipe
    print(f"fold {k+1}/{len(FOLDS)} done | {(time.time()-t0)/60:.1f} min")

for m in MODELS:
    print(f"{m:9s} out-of-fold AUC (repeat 1) = {roc_auc_score(y, PROB[m]):.4f}")
    pd.DataFrame(SHAP[m], columns=FEATS).assign(CustomerID=data[cfg["id_col"]], prob=PROB[m]) \
      .to_csv(os.path.join(OUT, f"shap_{m}.csv"), index=False)

# ---------------------------------------------------------------------
# 4. Global importance + stability (L5)
# ---------------------------------------------------------------------
glob_imp = pd.DataFrame({m: np.abs(SHAP[m]).mean(axis=0) for m in MODELS}, index=FEATS)
glob_imp = glob_imp.div(glob_imp.sum(axis=0), axis=1)          # share of total |SHAP|
glob_imp = glob_imp.sort_values(PRIMARY, ascending=False)
glob_imp.round(4).to_csv(os.path.join(TAB, "T5_global_importance.csv"))

# (a) across folds within the primary model
fold_rank = {}
for f in FOLDS:
    te = np.array(f["test_idx"])
    fold_rank[f"Fold {f['fold']+1}"] = np.abs(SHAP[PRIMARY][te]).mean(axis=0)
fold_rank = pd.DataFrame(fold_rank, index=FEATS)
tau_folds = [stats.kendalltau(fold_rank.iloc[:, i], fold_rank.iloc[:, j])[0]
             for i in range(fold_rank.shape[1]) for j in range(i + 1, fold_rank.shape[1])]
# (b) across models
tau_models = pd.DataFrame(index=MODELS, columns=MODELS, dtype=float)
for a in MODELS:
    for b in MODELS:
        tau_models.loc[a, b] = stats.kendalltau(glob_imp[a], glob_imp[b])[0]
# (c) local: top-3 feature overlap per customer between models (Jaccard)
def topk(sv, k=3): return np.argsort(-np.abs(sv), axis=1)[:, :k]
local_rows = []
for i, a in enumerate(MODELS):
    for b in MODELS[i + 1:]:
        ta, tb = topk(SHAP[a]), topk(SHAP[b])
        jac = [len(set(p) & set(q)) / len(set(p) | set(q)) for p, q in zip(ta, tb)]
        local_rows.append({"Pair": f"{a} vs {b}", "Global Kendall tau": tau_models.loc[a, b],
                           "Local top-3 Jaccard (mean)": np.mean(jac)})
local_rows.append({"Pair": f"{PRIMARY}: across {len(FOLDS)} folds", "Global Kendall tau": np.mean(tau_folds),
                   "Local top-3 Jaccard (mean)": np.nan})
stab = pd.DataFrame(local_rows)
stab.round(4).to_csv(os.path.join(TAB, "T6_explanation_stability.csv"), index=False)
print("\n=== Table 5: Global importance (share of total |SHAP|) ==="); display(glob_imp.round(4))
print("\n=== Table 6: Explanation stability ==="); display(stab.round(4))

# F11: global importance comparison
top = glob_imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
h = 0.8 / len(MODELS)
for i, m in enumerate(MODELS):
    ax.barh(np.arange(len(top)) + i * h, top[m] * 100, height=h, label=m)
ax.set_yticks(np.arange(len(top)) + h * (len(MODELS) - 1) / 2); ax.set_yticklabels(top.index)
ax.set_xlabel("Share of total |SHAP| (%)"); ax.legend(frameon=False)
ax.set_title("Global feature importance (top 15, out-of-fold SHAP)")
savefig("F11_global_importance")

# F12: SHAP beeswarm-style dot plot for primary model (numeric/binary features coloured by value)
top10 = glob_imp.index[:10][::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
for i, feat in enumerate(top10):
    j = FEATS.index(feat); sv = SHAP[PRIMARY][:, j]
    v = X[feat]
    col = (v.rank(pct=True).values if feat in NUM + BIN else
           pd.Series(pd.factorize(v)[0]).rank(pct=True).values)
    jitter = rng.normal(0, 0.12, len(sv))
    sc = ax.scatter(sv, i + jitter, c=col, cmap="coolwarm", s=3, alpha=0.5, rasterized=True)
ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10)
ax.set_xlabel("SHAP value (log-odds of churn)")
cb = plt.colorbar(sc, ax=ax, shrink=0.6); cb.set_label("Feature value (low → high / category code)")
ax.set_title(f"{PRIMARY}: local SHAP distribution (top 10)")
savefig("F12_shap_beeswarm")

# ---------------------------------------------------------------------
# 5. Optional LIME (primary model, churners with actionable stated reason)
#    LIME on the raw feature space with proper categorical handling
# ---------------------------------------------------------------------
churn_mask = (y == 1)
val_mask = churn_mask & data["Reason Group"].isin(REASON_TO_DOMAIN).values
val_idx = np.where(val_mask)[0]
print(f"\nValidation set: {len(val_idx)} churners with an actionable stated reason "
      f"(excluded: Uncontrollable + Unknown = {int(churn_mask.sum() - len(val_idx))})")

LIME = None
if RUN_LIME:
    cat_like = CAT
    enc = {c: {v: i for i, v in enumerate(sorted(X[c].unique()))} for c in cat_like}
    dec = {c: {i: v for v, i in enc[c].items()} for c in cat_like}
    Xcode = X.copy()
    for c in cat_like: Xcode[c] = X[c].map(enc[c])
    Xcode = Xcode.astype(float)
    cat_idx = [FEATS.index(c) for c in cat_like + BIN]
    cat_names = {FEATS.index(c): [dec[c][i] for i in range(len(dec[c]))] for c in cat_like}
    cat_names.update({FEATS.index(c): ["0", "1"] for c in BIN})

    def make_predict(pipe):
        def fn(arr):
            df = pd.DataFrame(arr, columns=FEATS)
            for c in cat_like: df[c] = df[c].round().clip(0, len(dec[c]) - 1).astype(int).map(dec[c])
            for c in BIN: df[c] = df[c].round().clip(0, 1).astype(int)
            return pipe.predict_proba(df)
        return fn

    LIME = np.zeros((len(data), len(FEATS)))
    t0 = time.time()
    fold_of = {i: f["fold"] for f in FOLDS for i in f["test_idx"]}
    for f in FOLDS:
        te = np.array(f["test_idx"]); tr = np.setdiff1d(np.arange(len(data)), te)
        explainer = LimeTabularExplainer(Xcode.iloc[tr].values, feature_names=FEATS, class_names=["stay", "churn"],
                                         categorical_features=cat_idx, categorical_names=cat_names,
                                         discretize_continuous=True, random_state=SEED, mode="classification")
        fn = make_predict(FITTED[(PRIMARY, f["fold"])])
        for i in np.intersect1d(te, val_idx):
            exp = explainer.explain_instance(Xcode.iloc[i].values, fn, num_features=len(FEATS),
                                             num_samples=LIME_SAMPLES, labels=(1,))
            for j, w in exp.as_map()[1]: LIME[i, j] = w
        print(f"LIME fold {f['fold']+1}/{len(FOLDS)} | {(time.time()-t0)/60:.1f} min")
    pd.DataFrame(LIME[val_idx], columns=FEATS).assign(CustomerID=data[cfg["id_col"]].values[val_idx]) \
      .to_csv(os.path.join(OUT, f"lime_{PRIMARY}.csv"), index=False)

# ---------------------------------------------------------------------
# 6. REASON VALIDATION (RQ2)
#    Domain score = share of a customer's POSITIVE (churn-pushing) attribution
#    that falls in each domain. Explanation "points to" the actionable domain
#    with the highest score. Compared with the customer's stated reason.
# ---------------------------------------------------------------------
true_dom = data.loc[val_idx, "Reason Group"].map(REASON_TO_DOMAIN).values

def domain_shares(att, dmap):
    pos = np.clip(att, 0, None)
    tot = pos.sum(axis=1, keepdims=True); tot[tot == 0] = 1
    return pd.DataFrame({d: pos[:, [FEATS.index(f) for f in fs]].sum(axis=1) / tot[:, 0]
                         for d, fs in dmap.items()})

def boot_ci(fn, n=N_BOOT):
    vals = []
    for _ in range(n):
        b = rng.integers(0, len(val_idx), len(val_idx))
        try: vals.append(fn(b))
        except ValueError: pass
    return np.percentile(vals, [2.5, 97.5])

def validate(att, label, dmap):
    sh = domain_shares(att[val_idx], dmap)
    pred = sh[ACTIONABLE].values.argmax(axis=1)
    pred_dom = np.array(ACTIONABLE)[pred]
    acc = (pred_dom == true_dom).mean()
    # permutation null: shuffle stated reasons across churners
    perm = np.array([(pred_dom == rng.permutation(true_dom)).mean() for _ in range(N_PERM)])
    p_perm = (np.sum(perm >= acc) + 1) / (N_PERM + 1)
    majority = pd.Series(true_dom).value_counts(normalize=True).iloc[0]
    row = {"Explanation": label, "Alignment acc.": acc, "Perm. null mean": perm.mean(),
           "Perm. p": p_perm, "Majority baseline": majority,
           "Cohen kappa": cohen_kappa_score(true_dom, pred_dom)}
    # threshold-free: does the domain score rank customers who state that reason higher?
    for d in ACTIONABLE:
        lab = (true_dom == d).astype(int)
        auc = roc_auc_score(lab, sh[d])
        lo, hi = boot_ci(lambda b: roc_auc_score(lab[b], sh[d].values[b]))
        row[f"AUC {d}"] = f"{auc:.3f} [{lo:.3f}, {hi:.3f}]"
    row["Non-actionable share"] = sh[["Relationship", "Demographic"]].sum(axis=1).mean()
    return row, sh, pred_dom, perm

results, store = [], {}
for m in MODELS:
    for mp_name, mp in [("primary", DOMAIN_MAP), ("alternative", DOMAIN_MAP_ALT)]:
        r, sh, pdm, perm = validate(SHAP[m], f"SHAP-{m} ({mp_name} map)", mp)
        results.append(r); store[(f"SHAP-{m}", mp_name)] = (sh, pdm, perm)
if LIME is not None:
    for mp_name, mp in [("primary", DOMAIN_MAP), ("alternative", DOMAIN_MAP_ALT)]:
        r, sh, pdm, perm = validate(LIME, f"LIME-{PRIMARY} ({mp_name} map)", mp)
        results.append(r); store[(f"LIME-{PRIMARY}", mp_name)] = (sh, pdm, perm)

val_tab = pd.DataFrame(results).set_index("Explanation")
num_cols = ["Alignment acc.", "Perm. null mean", "Perm. p", "Majority baseline", "Cohen kappa", "Non-actionable share"]
val_tab[num_cols] = val_tab[num_cols].astype(float).round(4)
val_tab.to_csv(os.path.join(TAB, "T7_reason_validation.csv"))
print("\n=== Table 7: Do explanations reflect stated churn reasons? ===")
display(val_tab)

# Domain shares by stated reason group (all churners, primary model, primary map)
sh_all = domain_shares(SHAP[PRIMARY][churn_mask], DOMAIN_MAP)
sh_all["Reason Group"] = data.loc[churn_mask, "Reason Group"].values
by_reason = sh_all.groupby("Reason Group")[ALL_DOMAINS].mean().mul(100)
by_reason = by_reason.loc[[g for g in ["Price", "Support", "Competitor", "Quality", "Uncontrollable", "Unknown"] if g in by_reason.index]]
by_reason.round(1).to_csv(os.path.join(TAB, "T8_domain_share_by_reason.csv"))
kw = {d: stats.kruskal(*[g[d].values for _, g in sh_all.groupby("Reason Group")]) for d in ALL_DOMAINS}
kw_tab = pd.DataFrame({d: {"H": r.statistic, "p": r.pvalue} for d, r in kw.items()}).T
kw_tab.round(5).to_csv(os.path.join(TAB, "T8b_kruskal_domain_by_reason.csv"))
print(f"\n=== Table 8: {PRIMARY} mean domain share (%) by stated reason ==="); display(by_reason.round(1))
print("Kruskal-Wallis (does domain share differ across reason groups?)"); display(kw_tab.round(5))

# ---------------------------------------------------------------------
# 7. Figures for the validation result
# ---------------------------------------------------------------------
# F13: heatmap — what explanations emphasise vs what customers said
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(by_reason, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of positive SHAP"}, ax=ax)
ax.set_title(f"Where {PRIMARY} explanations put churn credit, by customer's stated reason")
ax.set_xlabel("Explanation domain"); ax.set_ylabel("Stated reason group")
savefig("F13_domain_share_by_reason")

# F14: confusion matrix (primary model, primary map) + permutation null
sh, pdm, perm = store[(f"SHAP-{PRIMARY}", "primary")]
cm = pd.DataFrame(confusion_matrix(true_dom, pdm, labels=ACTIONABLE, normalize="true") * 100,
                  index=[f"Stated: {d}" for d in ACTIONABLE], columns=[f"Expl.: {d}" for d in ACTIONABLE])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.1]})
sns.heatmap(cm, annot=True, fmt=".1f", cmap="Oranges", cbar=False, ax=axes[0])
axes[0].set_title("Stated reason vs explanation (row %)")
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)
acc = (pdm == true_dom).mean()
axes[1].hist(perm, bins=40, color="#4A6FA5", alpha=0.8)
axes[1].axvline(acc, color="#C8553D", lw=2, label=f"Observed = {acc:.3f}")
axes[1].set_xlabel("Alignment accuracy under shuffled reasons"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Permutation null ({N_PERM} shuffles)"); axes[1].legend(frameon=False)
savefig("F14_alignment_confusion_permutation")

pd.Series({"n_validation": len(val_idx), "models": MODELS, "lime": LIME is not None}).to_json(os.path.join(OUT, "meta.json"))
print("\nSaved to", OUT, "->", sorted(os.listdir(OUT)))
print("\nHOW TO READ TABLE 7:")
print(" - Alignment acc. >> Perm. null mean and Perm. p < .05  -> explanations carry real reason signal")
print(" - AUC <domain> ~0.5 -> explanation cannot tell who left for that reason")
print(" - High 'Non-actionable share' -> explanations are dominated by risk markers (contract, tenure),")
print("   not by the causes customers report. BOTH outcomes are publishable findings.")

## Step 04: Stage 4: reason recoverability (why-model) and positive control


In [ ]:
# =====================================================================
# NB4 (single cell): Are stated reasons recoverable from the features?
#                    + Positive control for the validation protocol
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# Why: NB3 found explanations do NOT align with stated reasons.
#      Before claiming "XAI fails", we must separate two explanations:
#        (i)  the features carry reason signal but XAI misses it, or
#        (ii) the features carry no reason signal at all.
#   A. Univariate association: feature vs reason group (effect sizes)
#   B. Recoverability: a supervised reason classifier = upper bound
#   C. Positive control: synthetic reasons with KNOWN signal strength ->
#      does the NB3 protocol detect alignment when it truly exists?
#
# Input : NB1 + NB3 output in /kaggle/working
# Output: /kaggle/working/nb4/   RUNTIME: ~5-10 min
# =====================================================================

import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

N_PERM_CLF = 200     # permutations for the reason-classifier significance test
N_REP_CTRL = 20      # replicates per signal strength in the positive control
SIGNALS    = [0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0]
SEED       = 42

import os, json, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict, permutation_test_score
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, make_scorer
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

BASE = os.environ.get("NB_IN", WORK_DIR)
def find_file(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found. Run NB1-NB3 first.")

OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb4")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"{name}.png"), bbox_inches="tight"); plt.show()

cfg  = json.load(open(find_file("config.json")))
data = pd.read_csv(find_file("telco_clean.csv"))
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT

GROUPS = ["Price", "Support", "Competitor", "Quality"]
ch = data[(data[cfg["target"]] == 1) & data["Reason Group"].isin(GROUPS)].reset_index(drop=True)
Xc, rg = ch[FEATS], ch["Reason Group"].values
print(f"Actionable churners: {len(ch)} | groups: {pd.Series(rg).value_counts().to_dict()}")

# ---------------------------------------------------------------------
# A. Univariate association between each feature and reason group
#    numeric: Kruskal-Wallis + epsilon^2 ; categorical/binary: chi2 + Cramer's V
#    Holm correction over all features
# ---------------------------------------------------------------------
rows = []
for f in FEATS:
    if f in NUM:
        groups = [ch.loc[rg == g, f].values for g in GROUPS]
        H, p = stats.kruskal(*groups)
        eff = max(0.0, (H - len(GROUPS) + 1) / (len(ch) - len(GROUPS)))      # epsilon^2
        means = {g: ch.loc[rg == g, f].mean() for g in GROUPS}
        rows.append({"Feature": f, "Test": "Kruskal-Wallis", "Stat": H, "p": p,
                     "Effect": eff, "Effect type": "epsilon²", **{f"{g} mean/%": round(v, 2) for g, v in means.items()}})
    else:
        ct = pd.crosstab(ch[f], rg)
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        v = np.sqrt(chi2 / (len(ch) * (min(ct.shape) - 1)))
        top = ch[f].value_counts().index[0]
        pct = {g: (ch.loc[rg == g, f] == top).mean() * 100 for g in GROUPS}
        rows.append({"Feature": f + (f" [= {top}]" if f in CAT else ""), "Test": "Chi-square", "Stat": chi2, "p": p,
                     "Effect": v, "Effect type": "Cramér's V", **{f"{g} mean/%": round(x, 1) for g, x in pct.items()}})
uni = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
m = len(uni)
uni["p_holm"] = np.maximum.accumulate(np.minimum(1, uni["p"] * (m - np.arange(m))))
uni["Significant (Holm .05)"] = uni["p_holm"] < 0.05
uni.round(4).to_csv(os.path.join(TAB, "T9_univariate_feature_by_reason.csv"), index=False)
print("\n=== Table 9: Feature vs stated reason group (actionable churners) ===")
display(uni.round(4))
print(f"Features significant after Holm: {int(uni['Significant (Holm .05)'].sum())} / {m} | "
      f"max effect size = {uni['Effect'].max():.3f} (small effect: V≈0.1, eps²≈0.01)")

# ---------------------------------------------------------------------
# B. Recoverability: can ANY model predict the stated reason from features?
#    (upper bound on what an explanation could possibly reflect)
# ---------------------------------------------------------------------
def make_pre():
    return ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
CLFS = {
    "Multinomial LogReg": LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000),
    "Gradient Boosting":  HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                         class_weight="balanced", random_state=SEED),
}
le = LabelEncoder().fit(GROUPS)
y4 = le.transform(rg)
y3 = LabelEncoder().fit_transform(pd.Series(rg).replace({"Competitor": "Offering", "Quality": "Offering"}))
macro_auc = make_scorer(roc_auc_score, response_method="predict_proba", multi_class="ovr", average="macro")

rec_rows = []
for task, yy in [("4 reason groups", y4), ("3 domains (Price/Support/Offering)", y3)]:
    for name, clf in CLFS.items():
        pipe = Pipeline([("pre", make_pre()), ("clf", clf)])
        cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
        aucs, baccs = [], []
        for tr, te in cv.split(Xc, yy):
            p = pipe.fit(Xc.iloc[tr], yy[tr]).predict_proba(Xc.iloc[te])
            aucs.append(roc_auc_score(yy[te], p, multi_class="ovr", average="macro"))
            baccs.append(balanced_accuracy_score(yy[te], p.argmax(1)))
        t0 = time.time()
        score, perm_scores, p_perm = permutation_test_score(
            pipe, Xc, yy, scoring=macro_auc, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
            n_permutations=N_PERM_CLF, random_state=SEED, n_jobs=-1)
        rec_rows.append({"Task": task, "Model": name,
                         "Macro AUC": f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}",
                         "Balanced acc.": f"{np.mean(baccs):.3f} ± {np.std(baccs):.3f}",
                         "Chance bal. acc.": round(1 / len(np.unique(yy)), 3),
                         "Perm. null AUC (mean)": round(perm_scores.mean(), 3),
                         "Perm. p": round(p_perm, 4)})
        print(f"{task:36s} {name:20s} AUC {np.mean(aucs):.3f} | perm p {p_perm:.4f} | {(time.time()-t0)/60:.1f} min")
rec = pd.DataFrame(rec_rows)
rec.to_csv(os.path.join(TAB, "T10_reason_recoverability.csv"), index=False)
print("\n=== Table 10: Can stated reasons be predicted from the features at all? ==="); display(rec)

# ---------------------------------------------------------------------
# C. Positive control: inject reasons with KNOWN feature dependence,
#    then run the same SHAP-reason validation as NB3 (primary model, primary map)
# ---------------------------------------------------------------------
shap_file = None
for m_ in ["CatBoost", "XGBoost", "LogReg"]:          # primary model first
    try: shap_file = find_file(f"nb3/shap_{m_}.csv"); break
    except FileNotFoundError: pass
assert shap_file, "NB3 SHAP file not found"
model_name = os.path.basename(shap_file).replace("shap_", "").replace(".csv", "")
shap_all = pd.read_csv(shap_file)
idx = data.index[(data[cfg["target"]] == 1) & data["Reason Group"].isin(GROUPS)].values
SV = shap_all.loc[idx, FEATS].values

DOMAIN_MAP = {   # identical to NB3 primary map
    "Price":    ["Monthly Charges", "Total Charges", "Charge Gap"],
    "Support":  ["Tech Support", "Online Security", "Device Protection"],
    "Offering": ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines",
                 "Online Backup", "Phone Service", "Num Services"],
}
ACT = list(DOMAIN_MAP)
pos = np.clip(SV, 0, None); tot = pos.sum(1); tot[tot == 0] = 1
SHARE = np.column_stack([pos[:, [FEATS.index(f) for f in DOMAIN_MAP[d]]].sum(1) / tot for d in ACT])

def z(v): v = np.asarray(v, float); return (v - v.mean()) / (v.std() + 1e-9)
d_ = data.loc[idx]
# Latent "true cause" scores built from domain features (directions that raise churn risk)
LAT = np.column_stack([
    z(z(d_["Monthly Charges"]) + 0.5 * z(d_["Charge Gap"])),                                   # Price
    z((d_["Tech Support"] == "No").astype(int) + (d_["Online Security"] == "No").astype(int)),  # Support
    z((d_["Internet Service"] == "Fiber optic").astype(int) + 0.5 * z(d_["Num Services"])),     # Offering
])

def run_validation(true_idx):
    pred = SHARE.argmax(1)
    acc = (pred == true_idx).mean()
    aucs = [roc_auc_score((true_idx == j).astype(int), SHARE[:, j]) for j in range(3)]
    return acc, aucs

ctrl = []
for s in SIGNALS:
    for r in range(N_REP_CTRL):
        logits = s * LAT
        p = np.exp(logits - logits.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        true_idx = np.array([rng.choice(3, p=pi) for pi in p])
        acc, aucs = run_validation(true_idx)
        null = np.mean([(SHARE.argmax(1) == rng.permutation(true_idx)).mean() for _ in range(200)])
        ctrl.append({"signal": s, "rep": r, "acc": acc, "null_acc": null,
                     **{f"AUC {d}": a for d, a in zip(ACT, aucs)}})
ctrl = pd.DataFrame(ctrl)
ctrl_sum = ctrl.groupby("signal")[[f"AUC {d}" for d in ACT] + ["acc", "null_acc"]].mean()
ctrl_sum["Mean AUC"] = ctrl_sum[[f"AUC {d}" for d in ACT]].mean(1)
ctrl_sum["Acc. gain over null"] = ctrl_sum["acc"] - ctrl_sum["null_acc"]

# real data point
true_real = pd.Series(ch["Reason Group"]).replace({"Competitor": "Offering", "Quality": "Offering"}).map(
    {d: i for i, d in enumerate(ACT)}).values
acc_r, aucs_r = run_validation(true_real)
real_row = pd.Series({**{f"AUC {d}": a for d, a in zip(ACT, aucs_r)}, "Mean AUC": np.mean(aucs_r)}, name="Real IBM reasons")
ctrl_tab = pd.concat([ctrl_sum[[f"AUC {d}" for d in ACT] + ["Mean AUC", "Acc. gain over null"]],
                      real_row.to_frame().T])
ctrl_tab.round(3).to_csv(os.path.join(TAB, "T11_positive_control.csv"))
print(f"\n=== Table 11: Positive control ({model_name} SHAP, {N_REP_CTRL} replicates per signal) ===")
display(ctrl_tab.round(3))

# Figure: detection curve
fig, ax = plt.subplots(figsize=(7.5, 4.3))
cols = {"Price": "#C8553D", "Support": "#4A6FA5", "Offering": "#5B8C5A"}
for d in ACT:
    g = ctrl.groupby("signal")[f"AUC {d}"]
    ax.plot(SIGNALS, g.mean(), "o-", color=cols[d], label=f"{d} (synthetic)")
    ax.fill_between(SIGNALS, g.quantile(0.025), g.quantile(0.975), color=cols[d], alpha=0.12)
    ax.axhline(aucs_r[ACT.index(d)], color=cols[d], ls=":", lw=1.3)
ax.axhline(0.5, color="grey", lw=0.8)
ax.set_xlabel("Injected reason–feature signal strength"); ax.set_ylabel("Domain AUC of SHAP share")
ax.set_title("Positive control: protocol detects alignment when it exists\n(dotted = observed with real IBM reasons)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
savefig("F15_positive_control")

# Figure: univariate effect sizes
fig, ax = plt.subplots(figsize=(7.5, 5.5))
u = uni.sort_values("Effect")
ax.barh(u["Feature"], u["Effect"], color=np.where(u["Significant (Holm .05)"], "#C8553D", "#B0B7C3"))
ax.axvline(0.1, ls="--", color="grey", lw=0.8); ax.text(0.1, -0.9, " small (V=0.1)", fontsize=8, color="grey")
ax.set_xlabel("Effect size (Cramér's V / epsilon²)")
ax.set_title("Association between features and stated reason\n(red = significant after Holm)")
savefig("F16_univariate_effects")

print("\nSaved to", OUT, "->", sorted(os.listdir(OUT)))
print("\nHOW TO READ:")
print(" Table 10 Perm. p < .05 and AUC clearly > 0.5  -> reasons ARE (partly) recoverable -> XAI misses real signal")
print(" Table 10 AUC ~0.5, Perm. p > .05              -> features carry no reason signal -> SHAP cannot be read as 'cause'")
print(" Table 11: synthetic AUC rising with signal     -> the NB3 protocol is valid (has power)")

## Step 04b: Sources of the reason signal


In [ ]:
# =====================================================================
# NB4b (single cell): Where does the reason signal come from?
#   1. Recoverability with vs without geography (Latitude/Longitude)
#   2. Why-model feature importance (permutation importance, out-of-fold)
#   3. Price blind-spot diagnostic: does the churn model's Price-domain
#      SHAP share track Monthly Charges at all?
# Input : NB1, NB3 output in /kaggle/working     RUNTIME: ~5-8 min
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

SEED = 42
import os, json, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

BASE = os.environ.get("NB_IN", WORK_DIR)
def find_file(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb4")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})

cfg  = json.load(open(find_file("config.json")))
data = pd.read_csv(find_file("telco_clean.csv"))
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
GROUPS = ["Price", "Support", "Competitor", "Quality"]
ch = data[(data[cfg["target"]] == 1) & data["Reason Group"].isin(GROUPS)].reset_index(drop=True)
y4 = LabelEncoder().fit(GROUPS).transform(ch["Reason Group"])
GEO = ["Latitude", "Longitude"]

def make_pipe(num, model, bins=None, cats=None):
    bins = BIN if bins is None else bins; cats = CAT if cats is None else cats
    parts = [("num", StandardScaler(), num)]
    if bins: parts.append(("bin", "passthrough", bins))
    if cats: parts.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats))
    clf = (HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                          class_weight="balanced", random_state=SEED)
           if model == "GB" else LogisticRegression(C=0.5, class_weight="balanced", max_iter=3000))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", clf)])

CV = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(ch, y4))
def cv_auc(num, bins, cats, model):
    X = ch[num + bins + cats]
    return np.array([roc_auc_score(y4[te], make_pipe(num, model, bins, cats).fit(X.iloc[tr], y4[tr])
                                   .predict_proba(X.iloc[te]), multi_class="ovr", average="macro") for tr, te in CV])

# ---------------------------------------------------------------------
# 1. Recoverability with / without geography / geography only
# ---------------------------------------------------------------------
NUM_NOGEO = [f for f in NUM if f not in GEO]
sets = {"All features": (NUM, BIN, CAT),
        "Without Latitude/Longitude": (NUM_NOGEO, BIN, CAT),
        "Geography only": (GEO, [], [])}
rows, res = [], {}
for name, (num, bins, cats) in sets.items():
    for model in ["LogReg", "GB"]:
        a_ = cv_auc(num, bins, cats, model)
        res[(name, model)] = a_
        rows.append({"Feature set": name, "Model": "Gradient Boosting" if model == "GB" else "Multinomial LogReg",
                     "Macro AUC": f"{a_.mean():.3f} ± {a_.std():.3f}"})
geo_tab = pd.DataFrame(rows)
# paired test: all vs without geography (GB), same folds
d = res[("All features", "GB")] - res[("Without Latitude/Longitude", "GB")]
t, p = stats.ttest_rel(res[("All features", "GB")], res[("Without Latitude/Longitude", "GB")])
geo_tab.to_csv(os.path.join(TAB, "T12_recoverability_geo_ablation.csv"), index=False)
print("=== Table 12: Reason recoverability — role of geography (4 groups) ==="); display(geo_tab)
print(f"GB: all vs without geography  ΔAUC = {d.mean():.3f}, paired t = {t:.2f}, p = {p:.4f} (fold overlap -> indicative only)")

# ---------------------------------------------------------------------
# 2. Why-model importance (GB, all features): out-of-fold permutation importance
# ---------------------------------------------------------------------
X = ch[NUM + BIN + CAT]
imp = []
for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y4)):
    pipe = make_pipe(NUM, "GB").fit(X.iloc[tr], y4[tr])
    pi = permutation_importance(pipe, X.iloc[te], y4[te], n_repeats=10, random_state=SEED,
                                scoring=lambda est, Xv, yv: roc_auc_score(yv, est.predict_proba(Xv), multi_class="ovr", average="macro"))
    imp.append(pd.Series(pi.importances_mean, index=X.columns))
imp = pd.concat(imp, axis=1)
why_imp = pd.DataFrame({"Mean ΔAUC when permuted": imp.mean(1), "SD across folds": imp.std(1)}).sort_values(
    "Mean ΔAUC when permuted", ascending=False)
why_imp.round(4).to_csv(os.path.join(TAB, "T13_why_model_importance.csv"))
print("\n=== Table 13: What does the WHY-model use? (out-of-fold permutation importance) ===")
display(why_imp.head(10).round(4))

# churn-model importance for comparison (NB3)
for m_ in ["CatBoost", "XGBoost", "LogReg"]:
    try:
        shap_file = find_file(f"nb3/shap_{m_}.csv"); break
    except FileNotFoundError: pass
churn_model = os.path.basename(shap_file)[5:-4]
sv_all = pd.read_csv(shap_file)
feats = NUM + BIN + CAT
churn_imp = sv_all[feats].abs().mean(); churn_imp /= churn_imp.sum()
why_pos = why_imp["Mean ΔAUC when permuted"].clip(lower=0); why_pos /= why_pos.sum() if why_pos.sum() > 0 else 1
comp = pd.DataFrame({"Churn model (WHETHER)": churn_imp, "Why-model (WHY)": why_pos}).fillna(0)
comp = comp.loc[comp.max(1).sort_values(ascending=False).index[:12]]
rho = stats.spearmanr(churn_imp[feats], why_pos.reindex(feats).fillna(0))[0]

fig, ax = plt.subplots(figsize=(8, 5.5))
yy = np.arange(len(comp))[::-1]
ax.barh(yy + 0.2, comp.iloc[:, 0] * 100, height=0.4, color="#4A6FA5", label=f"Churn model ({churn_model} |SHAP| share)")
ax.barh(yy - 0.2, comp.iloc[:, 1] * 100, height=0.4, color="#C8553D", label="Why-model (permutation importance share)")
ax.set_yticks(yy); ax.set_yticklabels(comp.index); ax.set_xlabel("Share of importance (%)")
ax.set_title(f"Features that predict WHETHER ≠ features that inform WHY (Spearman ρ = {rho:.2f})")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "F17_whether_vs_why.png"), bbox_inches="tight"); plt.show()
print(f"Rank correlation churn-importance vs why-importance: Spearman rho = {rho:.3f}")

# ---------------------------------------------------------------------
# 3. Price blind spot: does Price-domain SHAP share follow Monthly Charges?
# ---------------------------------------------------------------------
idx = data.index[(data[cfg["target"]] == 1) & data["Reason Group"].isin(GROUPS)].values
SV = sv_all.loc[idx, feats].values
pos = np.clip(SV, 0, None); tot = pos.sum(1); tot[tot == 0] = 1
share = lambda fs: pos[:, [feats.index(f) for f in fs]].sum(1) / tot
mc = data.loc[idx, "Monthly Charges"].values
diag = pd.DataFrame({
    "Spearman ρ with Monthly Charges": {
        "Price share (MC + TC + Gap)": stats.spearmanr(mc, share(["Monthly Charges", "Total Charges", "Charge Gap"]))[0],
        "SHAP of Monthly Charges":     stats.spearmanr(mc, SV[:, feats.index("Monthly Charges")])[0],
        "SHAP of Total Charges":       stats.spearmanr(mc, SV[:, feats.index("Total Charges")])[0],
        "SHAP of Tenure Months":       stats.spearmanr(mc, SV[:, feats.index("Tenure Months")])[0],
    }})
diag.round(3).to_csv(os.path.join(TAB, "T14_price_blindspot.csv"))
print("\n=== Table 14: Price blind-spot diagnostic (actionable churners) ==="); display(diag.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(mc, SV[:, feats.index("Monthly Charges")], s=4, alpha=0.4, color="#4A6FA5")
axes[0].set(xlabel="Monthly Charges", ylabel="SHAP (Monthly Charges)", title="Own-feature attribution")
axes[1].scatter(mc, share(["Monthly Charges", "Total Charges", "Charge Gap"]), s=4, alpha=0.4, color="#C8553D")
axes[1].set(xlabel="Monthly Charges", ylabel="Price-domain share", title="Price-domain share of churn credit")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "F18_price_blindspot.png"), bbox_inches="tight"); plt.show()
print("\nSaved:", sorted(os.listdir(TAB)))

## Step 05: Stage 5: profit evaluation


In [ ]:
# =====================================================================
# NB5 (single cell): Profit evaluation of retention strategies
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# RQ4: Does choosing the retention offer from SHAP (as prior work does)
#      pay off, compared with a dedicated why-model and with an oracle?
# Solves: L6 (profit metrics ignore reasons / uncontrollable churn),
#         L7 (prescriptions never checked against the real cause),
#         L9 (calibration before value-based targeting)
#
# Customer-level realised profit on OUT-OF-FOLD predictions (repeat 1):
#   targeted churner   : gamma(offer, true reason) * CLTV - cost
#   targeted non-churner: - cost
#   Baseline offer = most common reason domain in training folds ("majority offer")
#   gamma = G_MATCH if offer matches true reason domain, G_MISS otherwise,
#           0 for Uncontrollable (Moved/Deceased), random-offer average for Unknown
# EMP: expected maximum profit with gamma_match ~ Beta (Monte-Carlo),
#      in the spirit of EMPC (Verbeke et al. 2012) with customer-level CLTV
#      (as in e-Profits) — not an exact re-implementation of either.
#
# Input : NB1-NB3 output in /kaggle/working   RUNTIME: ~3-6 min
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

G_MATCH, G_MISS = 0.40, 0.10        # offer success if matched / mismatched to true reason
COST            = 100               # contact + incentive cost per targeted customer
N_BOOT          = 1000
SEED            = 42

import os, json, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.isotonic import IsotonicRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

BASE = os.environ.get("NB_IN", WORK_DIR)
def find_file(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(name)
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb5")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, f"{name}.png"), bbox_inches="tight"); plt.show()

cfg    = json.load(open(find_file("config.json")))
data   = pd.read_csv(find_file("telco_clean.csv"))
splits = [s for s in json.load(open(find_file("nb2/outer_splits.json"))) if s["repeat"] == 0]
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT
y    = data[cfg["target"]].astype(int).values
CLTV = data[cfg["value_col"]].astype(float).values
N    = len(data)
fold_of = np.empty(N, int)
for s in splits: fold_of[s["test_idx"]] = s["fold"]

DOMAINS = ["Price", "Support", "Offering"]
OFFER = {"Price": "Discount / price lock", "Support": "Priority support call", "Offering": "Plan upgrade / speed boost"}
R2D = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
rg = data["Reason Group"].astype(object).values
true_dom = np.array([R2D.get(r, None) if isinstance(r, str) else None for r in rg], dtype=object)
uncontrollable = (rg == "Uncontrollable")
unknown        = (rg == "Unknown")

# ---------------------------------------------------------------------
# 1. Churn probabilities (out-of-fold, NB3) + cross-fitted isotonic calibration
# ---------------------------------------------------------------------
PROB = {}
for m in ["CatBoost", "XGBoost", "LogReg"]:
    try: PROB[m] = pd.read_csv(find_file(f"nb3/shap_{m}.csv"))["prob"].values
    except FileNotFoundError: pass
PRIMARY = next(iter(PROB))
print("Churn models:", list(PROB), "| primary:", PRIMARY)

def crossfit_isotonic(p):
    out = np.zeros_like(p)
    for k in np.unique(fold_of):
        tr, te = fold_of != k, fold_of == k        # calibrator never sees fold k labels
        out[te] = IsotonicRegression(out_of_bounds="clip").fit(p[tr], y[tr]).predict(p[te])
    return out
PCAL = {m: crossfit_isotonic(p) for m, p in PROB.items()}
cal_tab = pd.DataFrame({m: {"AUC raw": roc_auc_score(y, PROB[m]), "Brier raw": brier_score_loss(y, PROB[m]),
                            "Brier calibrated": brier_score_loss(y, PCAL[m]),
                            "Mean p raw": PROB[m].mean(), "Mean p calibrated": PCAL[m].mean(), "Churn rate": y.mean()}
                        for m in PROB}).T
cal_tab.round(4).to_csv(os.path.join(TAB, "T15_calibration.csv"))
print("\n=== Table 15: Calibration (cross-fitted isotonic) ==="); display(cal_tab.round(4))

# ---------------------------------------------------------------------
# 2. Offer-choice rules (all out-of-fold)
#    a) SHAP-guided: domain with largest positive SHAP share (what prior work implies)
#    b) Why-model: trained on churners' stated reasons in training folds
#    c) Oracle: true stated reason (upper bound)     d) Random offer (baseline)
# ---------------------------------------------------------------------
DMAP = {"Price": ["Monthly Charges", "Total Charges", "Charge Gap"],
        "Support": ["Tech Support", "Online Security", "Device Protection"],
        "Offering": ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines",
                     "Online Backup", "Phone Service", "Num Services"]}
sv = pd.read_csv(find_file(f"nb3/shap_{PRIMARY}.csv"))[FEATS].values
pos = np.clip(sv, 0, None)
shap_dom = np.array(DOMAINS)[np.column_stack([pos[:, [FEATS.index(f) for f in DMAP[d]]].sum(1) for d in DOMAINS]).argmax(1)]

why_dom = np.empty(N, dtype=object)
act = np.array([t is not None for t in true_dom]) & (y == 1)
for k in np.unique(fold_of):
    tr = (fold_of != k) & act; te = fold_of == k
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    # no class balancing: the profit-optimal offer is the MOST PROBABLE reason (maximises expected gamma)
    why = Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                                         random_state=SEED))])
    why.fit(data.loc[tr, FEATS], true_dom[tr].astype(str))
    why_dom[te] = why.predict(data.loc[te, FEATS])
rand_dom = rng.choice(DOMAINS, N)
# Majority offer: most common stated-reason domain among TRAINING-fold churners (strong naive baseline)
maj_dom = np.empty(N, dtype=object)
for k in np.unique(fold_of):
    tr = (fold_of != k) & act
    maj_dom[fold_of == k] = pd.Series(true_dom[tr]).value_counts().index[0]

acc_tab = pd.DataFrame({name: {"Offer matches stated reason (actionable churners)": (dom[act] == true_dom[act]).mean()}
                        for name, dom in [("SHAP-guided", shap_dom), ("Why-model", why_dom),
                                          ("Majority offer", maj_dom), ("Random", rand_dom)]}).T
acc_tab.loc["Oracle"] = 1.0
print("\nOffer-reason match rate:"); display(acc_tab.round(3))

# ---------------------------------------------------------------------
# 3. Profit engine
# ---------------------------------------------------------------------
def gamma_vec(offer_dom, g_match=G_MATCH, g_miss=G_MISS):
    g = np.where(offer_dom == true_dom, g_match, g_miss).astype(float)
    g[unknown] = (g_match + 2 * g_miss) / 3            # unknown cause: expected value of a random offer
    g[uncontrollable] = 0.0                             # moved / deceased cannot be retained
    g[y == 0] = 0.0                                     # non-churners: nothing to save
    return g

def profit_curve(score, offer_dom, g_match=G_MATCH, g_miss=G_MISS, cost=COST, idx=None):
    idx = np.arange(N) if idx is None else idx
    gain = gamma_vec(offer_dom, g_match, g_miss)[idx] * CLTV[idx] - cost
    order = np.argsort(-score[idx], kind="stable")
    return np.concatenate([[0], np.cumsum(gain[order])])   # profit when targeting top-k

BASELINE = "S2 Risk×CLTV + majority offer"
STRATS = {   # name: (ranking score, offer rule)
    "S1 Risk rank + majority offer":        (PCAL[PRIMARY],        maj_dom),
    BASELINE:                               (PCAL[PRIMARY] * CLTV, maj_dom),
    "S2r Risk×CLTV + random offer":         (PCAL[PRIMARY] * CLTV, rand_dom),
    "S3 Risk×CLTV + SHAP-guided offer":     (PCAL[PRIMARY] * CLTV, shap_dom),
    "S4 Risk×CLTV + why-model offer":       (PCAL[PRIMARY] * CLTV, why_dom),
    "S5 Risk×CLTV + oracle offer (bound)":  (PCAL[PRIMARY] * CLTV, true_dom),
    "S2u Risk×CLTV uncalibrated + majority":(PROB[PRIMARY] * CLTV, maj_dom),
}
curves = {s: profit_curve(sc, od) for s, (sc, od) in STRATS.items()}

def emp(score, offer_dom, n_mc=300):
    # EMP: average over gamma_match ~ Beta(6,14) (mean 0.30), gamma_miss = gamma_match/4
    vals = []
    for g in rng.beta(6, 14, n_mc):
        vals.append(profit_curve(score, offer_dom, g, g / 4).max())
    return np.mean(vals) / N

rows = []
for s, (sc, od) in STRATS.items():
    c = curves[s]; k = c.argmax()
    rows.append({"Strategy": s, "Max profit": c.max(), "Optimal % targeted": 100 * k / N,
                 "Profit per customer": c.max() / N, "EMP per customer": emp(sc, od)})
prof = pd.DataFrame(rows).set_index("Strategy")

# bootstrap CI for differences vs S2 (paired, same resampled customers)
def boot_diff(a, b):
    sa, oa = STRATS[a]; sb, ob = STRATS[b]; diffs = []
    for _ in range(N_BOOT):
        i = rng.integers(0, N, N)
        diffs.append(profit_curve(sa, oa, idx=i).max() - profit_curve(sb, ob, idx=i).max())
    return np.mean(diffs), *np.percentile(diffs, [2.5, 97.5])
for s in prof.index:
    if s == BASELINE: continue
    m_, lo, hi = boot_diff(s, BASELINE)
    prof.loc[s, "Δ vs S2 [95% CI]"] = f"{m_:,.0f} [{lo:,.0f}, {hi:,.0f}]"
prof.round(2).to_csv(os.path.join(TAB, "T16_profit_strategies.csv"))
print(f"\n=== Table 16: Retention profit (out-of-fold, {PRIMARY}, γ_match={G_MATCH}, γ_miss={G_MISS}, cost={COST}) ===")
display(prof.round(2))

# model comparison under the same strategy
mrows = []
for m in PROB:
    c = profit_curve(PCAL[m] * CLTV, maj_dom)
    mrows.append({"Churn model": m, "AUC": roc_auc_score(y, PROB[m]), "Max profit (S2)": c.max(),
                  "Optimal % targeted": 100 * c.argmax() / N})
mtab = pd.DataFrame(mrows).set_index("Churn model")
mtab.round(3).to_csv(os.path.join(TAB, "T17_profit_by_model.csv"))
print("\n=== Table 17: Does the AUC ranking carry over to profit? ==="); display(mtab.round(3))

# ---------------------------------------------------------------------
# 4. Sensitivity: when does reason-matching start to pay?
# ---------------------------------------------------------------------
G_GRID = [0.20, 0.30, 0.40, 0.50, 0.60]
C_GRID = [50, 100, 200, 400, 800]
sens = {}
for name, od in [("SHAP-guided", shap_dom), ("Why-model", why_dom), ("Oracle", true_dom)]:
    mat = pd.DataFrame(index=[f"γ_match={g}" for g in G_GRID], columns=[f"cost={c}" for c in C_GRID], dtype=float)
    for g in G_GRID:
        for c in C_GRID:
            base = profit_curve(PCAL[PRIMARY] * CLTV, maj_dom, g, G_MISS, c).max()
            alt  = profit_curve(PCAL[PRIMARY] * CLTV, od, g, G_MISS, c).max()
            mat.loc[f"γ_match={g}", f"cost={c}"] = 100 * (alt - base) / abs(base) if base != 0 else np.nan
    sens[name] = mat
    mat.round(1).to_csv(os.path.join(TAB, f"T18_sensitivity_{name}.csv"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
vmax = max(np.nanmax(np.abs(m.values)) for m in sens.values())
for ax, (name, mat) in zip(axes, sens.items()):
    sns.heatmap(mat, annot=True, fmt=".1f", cmap="RdBu", center=0, vmin=-vmax, vmax=vmax, cbar=False, ax=ax)
    ax.set_title(f"{name} offer: % profit gain vs majority offer")
savefig("F19_sensitivity")

# ---------------------------------------------------------------------
# 5. Profit curves
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 5))
style = {"S1": ("#9AA5B1", "--"), "S2": ("#4A6FA5", "-"), "S2r": ("#9AA5B1", ":"), "S3": ("#C8553D", "-"),
         "S4": ("#5B8C5A", "-"), "S5": ("#2F2F2F", ":"), "S2u": ("#4A6FA5", ":")}
x = np.arange(N + 1) / N * 100
for s, c in curves.items():
    col, ls = style[s.split()[0]]
    ax.plot(x, c / 1000, color=col, ls=ls, lw=1.6, label=s)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("% of customers targeted (ranked)"); ax.set_ylabel("Cumulative profit (thousands)")
ax.set_title("Retention profit curves (out-of-fold)"); ax.legend(frameon=False, fontsize=8)
ax.set_xlim(0, 60)
savefig("F20_profit_curves")

# ---------------------------------------------------------------------
# 6. Retention playbook: who to target and with which offer (why-model offer)
# ---------------------------------------------------------------------
k_opt = curves["S4 Risk×CLTV + why-model offer"].argmax()
tgt = np.argsort(-(PCAL[PRIMARY] * CLTV), kind="stable")[:k_opt]
seg = pd.DataFrame({"Offer (why-model)": [OFFER[d] for d in why_dom[tgt]],
                    "CLTV tier": pd.qcut(CLTV, 3, labels=["Low", "Mid", "High"])[tgt],
                    "Churn prob": PCAL[PRIMARY][tgt], "Churned": y[tgt]})
play = seg.groupby(["Offer (why-model)", "CLTV tier"]).agg(Customers=("Churned", "size"),
                                                           Mean_churn_prob=("Churn prob", "mean"),
                                                           Actual_churn_rate=("Churned", "mean")).round(3)
play.to_csv(os.path.join(TAB, "T19_retention_playbook.csv"))
print(f"\n=== Table 19: Retention playbook — {k_opt} customers targeted ({100*k_opt/N:.1f}%) ==="); display(play)

print("\nSaved to", OUT, "->", sorted(os.listdir(TAB)))
print("\nHOW TO READ:")
print(" Table 16: S3 (SHAP-guided) vs S2 (majority offer) -> does SHAP-based prescription add profit?")
print("           S4 (why-model) and S5 (oracle bound) -> how much a real reason signal is worth")
print(" Table 17: does a higher AUC also give higher profit?")
print(" F19: under which offer-effectiveness / cost settings reason-matching pays off")

## Step 06: Final tables and figures


In [ ]:
# =====================================================================
# NB6 (single cell): FINAL manuscript tables + figures
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# Reads NB1-NB5 outputs from /kaggle/working and produces:
#   final/tables/   Table_1..8 (main) + Table_A1..A9 (appendix) as CSV
#   final/manuscript_tables.xlsx   (one sheet per table, caption in row 1)
#   final/figures/  Fig_2..8 as PNG (600 dpi) + PDF (vector)
#   final/captions.md  draft captions with the actual numbers
# Also runs the ROBUSTNESS check (Total Charges moved to Relationship domain)
# RUNTIME: ~5-8 min.  Fig 1 (framework diagram) must be drawn by hand (draw.io).
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

N_BOOT, N_PERM, N_REP_CTRL, SEED = 1000, 2000, 20, 42
SIGNALS = [0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0]

import os, json, glob, re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.isotonic import IsotonicRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

BASE = os.environ.get("NB_IN", WORK_DIR)
def ff(name, required=True):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    if required: raise FileNotFoundError(f"{name} not found - run NB1-NB5 first")
    return None
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "final")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)

# ---------- journal style ----------
C = {"blue": "#3B5B92", "red": "#C0392B", "green": "#4E8A55", "grey": "#8C939D", "dark": "#2B2B2B",
     "light": "#DCE3EE", "orange": "#D98C2B"}
MODEL_COL = {"CatBoost": C["red"], "XGBoost": C["orange"], "LightGBM": C["green"],
             "RandomForest": C["blue"], "LogReg": C["dark"], "MLP": C["grey"]}
DOM_COL = {"Price": C["red"], "Support": C["blue"], "Offering": C["green"]}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
                     "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                     "savefig.dpi": 600, "pdf.fonttype": 42})
def panel(ax, letter): ax.text(-0.12, 1.06, f"({letter})", transform=ax.transAxes, fontsize=11, fontweight="bold")
def savefig(name):
    plt.tight_layout()
    for ext in ["png", "pdf"]: plt.savefig(os.path.join(FIG, f"{name}.{ext}"), bbox_inches="tight")
    plt.show()

TABLES, CAPTIONS = {}, {}
def table(key, df, caption, index=True):
    TABLES[key] = (df, caption, index); CAPTIONS[key] = caption
    df.to_csv(os.path.join(TAB, f"{key}.csv"), index=index)
    print(f"\n=== {key}: {caption[:110]}{'...' if len(caption) > 110 else ''}"); display(df)
fmt = lambda m, s, d=3: f"{m:.{d}f} ± {s:.{d}f}"

# ---------- load ----------
cfg  = json.load(open(ff("config.json")))
data = pd.read_csv(ff("telco_clean.csv"))
TARGET, VALUE = cfg["target"], cfg["value_col"]
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT
y, CLTV, N = data[TARGET].astype(int).values, data[VALUE].astype(float).values, len(data)
splits = [s for s in json.load(open(ff("nb2/outer_splits.json"))) if s["repeat"] == 0]
fold_of = np.empty(N, int)
for s in splits: fold_of[s["test_idx"]] = s["fold"]
SHAPM = {}
for m in ["CatBoost", "XGBoost", "LogReg"]:
    p = ff(f"nb3/shap_{m}.csv", required=False)
    if p: SHAPM[m] = pd.read_csv(p)
PRIMARY = next(iter(SHAPM))
lime_p = ff(f"nb3/lime_{PRIMARY}.csv", required=False)
LIME = pd.read_csv(lime_p) if lime_p else None
print("Explained models:", list(SHAPM), "| LIME:", LIME is not None)

GROUPS4 = ["Price", "Support", "Competitor", "Quality"]
R2D = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
DOMS = ["Price", "Support", "Offering"]
rg = data["Reason Group"].astype(object).values
act = (y == 1) & np.isin(rg, GROUPS4)
val_idx = np.where(act)[0]
true_dom = np.array([R2D.get(r) if isinstance(r, str) else None for r in rg], dtype=object)

MAPS = {
 "Primary": {"Price": ["Monthly Charges", "Total Charges", "Charge Gap"],
             "Support": ["Tech Support", "Online Security", "Device Protection"],
             "Offering": ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines", "Online Backup",
                          "Phone Service", "Num Services"],
             "Relationship": ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"],
             "Demographic": ["Male", "Senior Citizen", "Partner", "Dependents", "Latitude", "Longitude"]},
}
MAPS["Alternative"] = {"Price": MAPS["Primary"]["Price"] + ["Internet Service"], "Support": ["Tech Support"],
                       "Offering": ["Online Security", "Device Protection", "Streaming TV", "Streaming Movies",
                                    "Multiple Lines", "Online Backup", "Phone Service", "Num Services"],
                       "Relationship": MAPS["Primary"]["Relationship"], "Demographic": MAPS["Primary"]["Demographic"]}
MAPS["Robustness"] = {**MAPS["Primary"], "Price": ["Monthly Charges", "Charge Gap"],
                      "Relationship": MAPS["Primary"]["Relationship"] + ["Total Charges"]}   # Total Charges ~ tenure proxy
for mp in MAPS.values(): assert sorted(f for v in mp.values() for f in v) == sorted(FEATS)

def shares(att, mp):
    pos = np.clip(att, 0, None); tot = pos.sum(1, keepdims=True); tot[tot == 0] = 1
    return pd.DataFrame({d: pos[:, [FEATS.index(f) for f in fs]].sum(1) / tot[:, 0] for d, fs in mp.items()})

# =====================================================================
# TABLE 1  Descriptives      TABLE 2  Leakage audit      TABLE 3  Reason taxonomy
# =====================================================================
t1 = pd.read_csv(ff("tables/T1_descriptives.csv"), index_col=0)
t1 = t1.drop(index=[i for i in t1.index if i.startswith("Avg Monthly Spend")], errors="ignore")
table("Table_1", t1, "Descriptive statistics by churn status (mean ± SD for continuous variables; % for binary variables).")

leak = pd.read_csv(ff("tables/T_leakage_audit.csv"), index_col=0)
leak["Decision"] = ["Excluded (target copy)", "Excluded (vendor model score)",
                    "Excluded from features; used only as validation ground truth", "Excluded from features; used in profit layer"][:len(leak)]
table("Table_2", leak, "Leakage audit: discrimination achieved by each suspicious column on its own (standalone AUC) "
                       "and its treatment in this study.")

rg_rows = []
groups_all = data.loc[y == 1, "Reason Group"].value_counts()
for g in ["Competitor", "Quality", "Support", "Price", "Unknown", "Uncontrollable"]:
    if g not in groups_all: continue
    reasons = data.loc[(y == 1) & (data["Reason Group"] == g), "Churn Reason"].value_counts()
    rg_rows.append({"Reason group": g, "Stated reasons (n)": "; ".join(f"{r} ({n})" for r, n in reasons.items()),
                    "n": int(groups_all[g]), "% of churners": round(100 * groups_all[g] / (y == 1).sum(), 1),
                    "Explanation domain": R2D.get(g, "— (excluded from validation)"),
                    "Mean CLTV": round(data.loc[(y == 1) & (data["Reason Group"] == g), VALUE].mean(), 0)})
table("Table_3", pd.DataFrame(rg_rows).set_index("Reason group"),
      "Taxonomy of stated churn reasons, their mapping to explanation domains, and customer value.")

# =====================================================================
# TABLE 4  Predictive performance (nested CV) + stats
# =====================================================================
fm = pd.read_csv(ff("nb2/fold_metrics.csv"))
auc_mat = fm.pivot(index="Fold", columns="Model", values="AUC")
order = auc_mat.mean().sort_values(ascending=False).index.tolist()
best = order[0]; J = auc_mat.shape[0]
n_te = len(splits[0]["test_idx"]); n_tr = N - n_te
ranks = auc_mat.rank(axis=1, ascending=False).mean()
fr = stats.friedmanchisquare(*[auc_mat[c] for c in auc_mat]) if auc_mat.shape[1] >= 3 else None
praw = {}
for mdl in order[1:]:
    d = auc_mat[best] - auc_mat[mdl]; v = (1 / J + n_te / n_tr) * d.var(ddof=1)
    praw[mdl] = 2 * stats.t.sf(abs(d.mean() / np.sqrt(v)), J - 1) if v > 0 else 1.0
ps = pd.Series(praw).sort_values(); mm = len(ps)
holm = pd.Series(np.maximum.accumulate(np.minimum(1, ps.values * (mm - np.arange(mm)))), index=ps.index)
rows = []
for mdl in order:
    g = fm[fm.Model == mdl]
    rows.append({"Model": mdl, **{k: fmt(g[k].mean(), g[k].std()) for k in ["AUC", "PR_AUC", "F1", "Recall", "Brier"]},
                 "Mean rank": round(ranks[mdl], 2),
                 f"p (Holm) vs {best}": "—" if mdl == best else f"{holm[mdl]:.3f}"})
t4 = pd.DataFrame(rows).set_index("Model").rename(columns={"PR_AUC": "PR-AUC"})
fr_txt = f"Friedman χ² = {fr.statistic:.2f}, p = {fr.pvalue:.1e}" if fr else ""
table("Table_4", t4, f"Out-of-sample performance under nested {J}-fold repeated cross-validation (mean ± SD over outer folds). "
                     f"{fr_txt}. p-values: Nadeau–Bengio corrected resampled t-test vs the best model, Holm-adjusted. "
                     "Threshold-based metrics at 0.5; the MLP is not class-weighted.")

# =====================================================================
# TABLE 5  Explanation stability (global + local)
# =====================================================================
SV = {m: df[FEATS].values for m, df in SHAPM.items()}
mods = list(SV)
def topk(a, k=3): return np.argsort(-np.abs(a), 1)[:, :k]
gi = pd.DataFrame({m: np.abs(SV[m]).mean(0) for m in mods}, index=FEATS)
st_rows = []
for i, a in enumerate(mods):
    for b in mods[i + 1:]:
        ja = np.array([len(set(p) & set(q)) / len(set(p) | set(q)) for p, q in zip(topk(SV[a]), topk(SV[b]))])
        st_rows.append({"Comparison": f"{a} vs {b}", "Global Kendall τ": round(stats.kendalltau(gi[a], gi[b])[0], 3),
                        "Local top-3 Jaccard (mean)": round(ja.mean(), 3),
                        "% customers with identical top-3": round(100 * (ja == 1).mean(), 1),
                        "ΔAUC (out-of-fold)": round(abs(roc_auc_score(y, SHAPM[a]["prob"]) - roc_auc_score(y, SHAPM[b]["prob"])), 4)})
fold_imp = pd.DataFrame({k: np.abs(SV[PRIMARY][fold_of == k]).mean(0) for k in np.unique(fold_of)}, index=FEATS)
taus = [stats.kendalltau(fold_imp[i], fold_imp[j])[0] for i in fold_imp for j in fold_imp if i < j]
st_rows.append({"Comparison": f"{PRIMARY} across {fold_imp.shape[1]} folds", "Global Kendall τ": round(np.mean(taus), 3),
                "Local top-3 Jaccard (mean)": np.nan, "% customers with identical top-3": np.nan, "ΔAUC (out-of-fold)": np.nan})
table("Table_5", pd.DataFrame(st_rows).set_index("Comparison"),
      "Stability of SHAP explanations across models and folds. Near-identical accuracy does not imply near-identical "
      "customer-level explanations (Rashomon effect).")

# =====================================================================
# TABLE 6  Explanation–reason validation (3 maps incl. robustness)
# =====================================================================
td = true_dom[val_idx]
def boot_auc(lab, sc):
    v = []
    for _ in range(N_BOOT):
        b = rng.integers(0, len(lab), len(lab))
        if lab[b].min() != lab[b].max(): v.append(roc_auc_score(lab[b], sc[b]))
    return np.percentile(v, [2.5, 97.5])
VAL, FOREST = [], []
expl = [(f"SHAP-{m}", SV[m][val_idx]) for m in mods]
if LIME is not None:
    lime_mat = np.zeros((N, len(FEATS)))
    pos_map = {cid: i for i, cid in enumerate(data[cfg["id_col"]])}
    for _, r in LIME.iterrows(): lime_mat[pos_map[r[cfg["id_col"]]]] = r[FEATS].values
    expl.append((f"LIME-{PRIMARY}", lime_mat[val_idx]))
for mp_name, mp in MAPS.items():
    for name, att in expl:
        sh = shares(att, mp)
        pred = np.array(DOMS)[sh[DOMS].values.argmax(1)]
        acc = (pred == td).mean()
        perm = np.array([(pred == rng.permutation(td)).mean() for _ in range(N_PERM)])
        row = {"Explanation": name, "Domain map": mp_name, "Alignment acc.": round(acc, 3),
               "Permutation p": round((np.sum(perm >= acc) + 1) / (N_PERM + 1), 3),
               "Majority baseline": round(pd.Series(td).value_counts(normalize=True).iloc[0], 3),
               "Cohen κ": round(cohen_kappa_score(td, pred), 3)}
        for d in DOMS:
            lab = (td == d).astype(int); auc = roc_auc_score(lab, sh[d].values); lo, hi = boot_auc(lab, sh[d].values)
            row[f"AUC {d} [95% CI]"] = f"{auc:.3f} [{lo:.3f}, {hi:.3f}]"
            FOREST.append({"Explanation": name, "Map": mp_name, "Domain": d, "AUC": auc, "lo": lo, "hi": hi})
        row["Non-actionable share"] = round(sh[["Relationship", "Demographic"]].sum(1).mean(), 3)
        VAL.append(row)
t6 = pd.DataFrame(VAL).set_index(["Domain map", "Explanation"])
FOREST = pd.DataFrame(FOREST)
table("Table_6", t6, f"Agreement between explanations and customers' stated churn reasons (n = {len(val_idx)} churners with an "
                     f"actionable reason). Domain AUC: does the domain's share of positive attribution rank customers who "
                     f"stated that reason higher (0.5 = no information)? 95% CIs from {N_BOOT} bootstrap resamples; "
                     f"permutation p from {N_PERM} label shuffles. 'Robustness' moves Total Charges to the Relationship domain.")

# =====================================================================
# TABLE 7  Reason recoverability (incl. geography ablation)
# =====================================================================
t10 = pd.read_csv(ff("nb4/tables/T10_reason_recoverability.csv"))
t12 = pd.read_csv(ff("nb4/tables/T12_recoverability_geo_ablation.csv"))
t7a = t10.assign(**{"Feature set": "All features"})[["Task", "Feature set", "Model", "Macro AUC", "Balanced acc.", "Chance bal. acc.", "Perm. p"]]
t7b = t12[t12["Feature set"] != "All features"].assign(Task="4 reason groups")
t7 = pd.concat([t7a, t7b], ignore_index=True).fillna("—").set_index(["Task", "Feature set", "Model"])
table("Table_7", t7, "Can stated reasons be predicted from customer features at all? Supervised 'why-model' trained only on "
                     "churners (repeated stratified 5-fold CV; permutation test with 200 label shuffles). The signal "
                     "disappears without geography.")

# =====================================================================
# TABLE 8  Profit of retention strategies
# =====================================================================
t16 = pd.read_csv(ff("nb5/tables/T16_profit_strategies.csv"), index_col=0)
base_name = [s for s in t16.index if s.startswith("S2 ")][0]
base = t16.loc[base_name, "Max profit"]
t8 = pd.DataFrame({"Max profit": t16["Max profit"].round(0).astype(int),
                   "% change vs S2": ((t16["Max profit"] - base) / abs(base) * 100).round(1),
                   "Δ vs S2 [95% CI]": t16.get("Δ vs S2 [95% CI]", pd.Series(index=t16.index, dtype=str)).fillna("—"),
                   "Optimal % targeted": t16["Optimal % targeted"].round(1),
                   "Profit per customer": t16["Profit per customer"].round(1),
                   "EMP per customer": t16["EMP per customer"].round(1)})
table("Table_8", t8, "Realised retention profit on out-of-fold predictions (offer success 0.40 if the offer matches the stated "
                     "reason, 0.10 otherwise; 0 for uncontrollable churn; cost 100 per contact). Paired bootstrap CIs "
                     "(1000 resamples) for the difference to the majority-offer baseline S2.")

# =====================================================================
# APPENDIX TABLES
# =====================================================================
for key, path, cap, idx in [
    ("Table_A1", "nb2/tables/T4_smote_ablation.csv", "In-fold SMOTE versus class weighting (first CV repeat).", True),
    ("Table_A2", "nb4/tables/T9_univariate_feature_by_reason.csv", "Univariate association of each feature with the stated reason group (Holm-adjusted).", False),
    ("Table_A3", "nb3/tables/T5_global_importance.csv", "Global feature importance (share of total |SHAP|) by model.", True),
    ("Table_A4", "nb3/tables/T8_domain_share_by_reason.csv", f"Mean share (%) of positive {PRIMARY} attribution per domain, by stated reason group.", True),
    ("Table_A5", "nb5/tables/T15_calibration.csv", "Probability calibration before and after cross-fitted isotonic regression.", True),
    ("Table_A6", "nb5/tables/T17_profit_by_model.csv", "Retention profit (strategy S2) by churn model.", True),
    ("Table_A7", "nb5/tables/T18_sensitivity_SHAP-guided.csv", "Sensitivity: % profit change of SHAP-guided offers vs majority offer across offer-success and cost settings.", True),
    ("Table_A8", "nb5/tables/T19_retention_playbook.csv", "Retention playbook at the profit-optimal targeting depth.", False),
]:
    p = ff(path, required=False)
    if p: table(key, pd.read_csv(p, index_col=0 if idx else None), cap, index=idx)

# =====================================================================
# Positive control (primary + robustness map) -> Table A9 + Fig 6
# =====================================================================
d_ = data.loc[val_idx]
z = lambda v: (np.asarray(v, float) - np.mean(v)) / (np.std(v) + 1e-9)
LAT = np.column_stack([z(z(d_["Monthly Charges"]) + 0.5 * z(d_["Charge Gap"])),
                       z((d_["Tech Support"] == "No").astype(int) + (d_["Online Security"] == "No").astype(int)),
                       z((d_["Internet Service"] == "Fiber optic").astype(int) + 0.5 * z(d_["Num Services"]))])
PC = []
for mp_name in ["Primary", "Robustness"]:
    SH = shares(SV[PRIMARY][val_idx], MAPS[mp_name])[DOMS].values
    for s in SIGNALS:
        for r in range(N_REP_CTRL):
            lg = s * LAT; p = np.exp(lg - lg.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
            t = (p.cumsum(1) > rng.random((len(p), 1))).argmax(1)
            PC += [{"Map": mp_name, "signal": s, "Domain": d, "AUC": roc_auc_score((t == j).astype(int), SH[:, j])}
                   for j, d in enumerate(DOMS)]
PC = pd.DataFrame(PC)
a9 = PC.groupby(["Map", "signal", "Domain"])["AUC"].mean().unstack("Domain")[DOMS].round(3)
table("Table_A9", a9, f"Positive control: mean domain AUC of {PRIMARY} SHAP shares when synthetic reasons with known "
                      f"feature dependence are injected ({N_REP_CTRL} replicates per signal level).")

# =====================================================================
# FIGURES
# =====================================================================
# ---- Fig 2: data overview ----
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.3]})
overall = y.mean() * 100
for ax, col, L in zip(axes[:2], ["Contract", "Internet Service"], "ab"):
    r = data.groupby(col)[TARGET].mean().mul(100).sort_values()
    ax.barh(r.index, r.values, color=C["blue"], height=0.6)
    for i, v in enumerate(r.values): ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=8)
    ax.axvline(overall, ls="--", lw=0.8, color=C["dark"]); ax.set_xlim(0, r.max() * 1.25)
    ax.set_xlabel("Churn rate (%)"); ax.set_title(f"Churn by {col.lower()}"); panel(ax, L)
g = pd.Series({k: v["n"] for k, v in pd.DataFrame(rg_rows).set_index("Reason group").T.to_dict().items()}).sort_values()
cols = [C["grey"] if k in ["Unknown", "Uncontrollable"] else C["red"] for k in g.index]
axes[2].barh(g.index, g.values, color=cols, height=0.6)
for i, v in enumerate(g.values): axes[2].text(v + 8, i, f"{v} ({100*v/(y==1).sum():.0f}%)", va="center", fontsize=8)
axes[2].set_xlim(0, g.max() * 1.3); axes[2].set_xlabel("Churned customers")
axes[2].set_title("Stated churn reason groups"); panel(axes[2], "c")
savefig("Fig_2_data_overview")

# ---- Fig 3: predictive performance ----
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.1, 1]})
bp = axes[0].boxplot([auc_mat[m] for m in order], vert=False, patch_artist=True, widths=0.55,
                     medianprops=dict(color=C["dark"]))
for patch, m in zip(bp["boxes"], order): patch.set_facecolor(MODEL_COL.get(m, C["grey"])); patch.set_alpha(0.55)
axes[0].set_yticks(range(1, len(order) + 1)); axes[0].set_yticklabels(order); axes[0].invert_yaxis()
axes[0].set_xlabel("AUC (outer folds)"); axes[0].set_title(f"Discrimination across {J} outer folds"); panel(axes[0], "a")
k_mod = len(order); Q = {3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949}
CD = Q.get(k_mod, 2.85) * np.sqrt(k_mod * (k_mod + 1) / (6 * J))
ax = axes[1]; rk = ranks.sort_values()
ax.hlines(0, 1, k_mod, color=C["dark"], lw=1)
for i, (m, r) in enumerate(rk.items()):
    ax.plot(r, 0, "o", color=MODEL_COL.get(m, C["grey"]), ms=7)
    yy = 0.35 + 0.28 * (i % 3)
    ax.annotate(f"{m}\n{r:.2f}", (r, 0), xytext=(r, yy), ha="center", fontsize=7.5, arrowprops=dict(arrowstyle="-", lw=0.5))
ax.plot([1, 1 + CD], [-0.45, -0.45], color=C["red"], lw=3); ax.text(1 + CD / 2, -0.75, f"CD = {CD:.2f}", ha="center", fontsize=8)
ax.set_xlim(0.7, k_mod + 0.3); ax.set_ylim(-1.0, 1.3); ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.set_xlabel("Mean rank (lower is better)"); ax.set_title("Critical-difference diagram (Nemenyi, α = .05)"); panel(ax, "b")
savefig("Fig_3_performance")

# ---- Fig 4: explanations & Rashomon ----
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.35, 1]})
gis = gi.div(gi.sum(0), axis=1).sort_values(PRIMARY, ascending=False).head(12).iloc[::-1]
h = 0.8 / len(mods)
for i, m in enumerate(mods):
    axes[0].barh(np.arange(len(gis)) + (i - (len(mods) - 1) / 2) * h, gis[m] * 100, height=h, color=MODEL_COL[m], label=m)
axes[0].set_yticks(range(len(gis))); axes[0].set_yticklabels(gis.index)
axes[0].set_xlabel("Share of total |SHAP| (%)"); axes[0].legend(frameon=False, loc="lower right")
axes[0].set_title("Global importance (top 12)"); panel(axes[0], "a")
M = pd.DataFrame(np.eye(len(mods)), index=mods, columns=mods)
for r in st_rows[:-1]:
    a, b = r["Comparison"].split(" vs ")
    M.loc[a, b] = r["Global Kendall τ"]; M.loc[b, a] = r["Local top-3 Jaccard (mean)"]
sns.heatmap(M, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=axes[1], annot_kws={"size": 9})
axes[1].set_title("Agreement: upper = global τ, lower = local Jaccard"); panel(axes[1], "b")
savefig("Fig_4_explanations_rashomon")

# ---- Fig 5: KEY — explanations vs stated reasons ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})
sh_all = shares(SV[PRIMARY][y == 1], MAPS["Primary"]); sh_all["g"] = rg[y == 1]
br = sh_all.groupby("g")[list(MAPS["Primary"])].mean().mul(100)
br = br.loc[[g for g in ["Price", "Support", "Competitor", "Quality", "Uncontrollable", "Unknown"] if g in br.index]]
sns.heatmap(br, annot=True, fmt=".1f", cmap="Greys", cbar_kws={"label": "% of positive SHAP"}, ax=axes[0], annot_kws={"size": 8})
axes[0].set_ylabel("Stated reason group"); axes[0].set_xlabel("Explanation domain")
axes[0].set_title(f"{PRIMARY}: same explanation profile for every reason"); panel(axes[0], "a")
fp = FOREST[FOREST.Map == "Primary"].reset_index(drop=True)
ylab, ypos = [], []
for i, (_, r) in enumerate(fp.iterrows()):
    yv = len(fp) - i
    axes[1].errorbar(r.AUC, yv, xerr=[[r.AUC - r.lo], [r.hi - r.AUC]], fmt="o", color=DOM_COL[r.Domain], ms=4, capsize=2, lw=1)
    ylab.append(f"{r.Explanation} · {r.Domain}"); ypos.append(yv)
axes[1].axvline(0.5, color=C["dark"], ls="--", lw=0.8)
axes[1].set_yticks(ypos); axes[1].set_yticklabels(ylab, fontsize=7)
axes[1].set_xlabel("Domain AUC vs stated reason (95% CI)"); axes[1].set_xlim(0.40, 0.65)
axes[1].set_title("No explanation identifies the stated reason"); panel(axes[1], "b")
savefig("Fig_5_reason_validation")

# ---- Fig 6: positive control ----
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
real = FOREST[(FOREST.Explanation == f"SHAP-{PRIMARY}")]
for ax, mp_name, L in zip(axes, ["Primary", "Robustness"], "ab"):
    sub = PC[PC.Map == mp_name]
    for d in DOMS:
        g = sub[sub.Domain == d].groupby("signal")["AUC"]
        ax.plot(SIGNALS, g.mean(), "o-", color=DOM_COL[d], ms=3.5, lw=1.4, label=d)
        ax.fill_between(SIGNALS, g.quantile(.025), g.quantile(.975), color=DOM_COL[d], alpha=0.12)
        rv = real[(real.Map == mp_name) & (real.Domain == d)]["AUC"]
        if len(rv): ax.axhline(rv.iloc[0], color=DOM_COL[d], ls=":", lw=1.1)
    ax.axhline(0.5, color=C["grey"], lw=0.8)
    ax.set_xlabel("Injected reason–feature signal strength")
    ax.set_title(f"{mp_name} domain map" + (" (Total Charges → Relationship)" if mp_name == "Robustness" else ""))
    panel(ax, L)
axes[0].set_ylabel("Domain AUC"); axes[0].legend(frameon=False, title="Domain (dotted = real reasons)", loc="upper left")
savefig("Fig_6_positive_control")

# ---- Fig 7: whether vs why ----
t13 = pd.read_csv(ff("nb4/tables/T13_why_model_importance.csv"), index_col=0).iloc[:, 0].clip(lower=0)
why = t13 / t13.sum() if t13.sum() > 0 else t13
whe = gi[PRIMARY] / gi[PRIMARY].sum()
cmp_ = pd.DataFrame({"whether": whe, "why": why.reindex(FEATS).fillna(0)})
rho = stats.spearmanr(cmp_["whether"], cmp_["why"])[0]
top = cmp_.loc[cmp_.max(1).sort_values(ascending=False).index[:12]].iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
yy = np.arange(len(top))
axes[0].barh(yy + 0.2, top["whether"] * 100, 0.4, color=C["blue"], label=f"Churn model ({PRIMARY}, |SHAP|)")
axes[0].barh(yy - 0.2, top["why"] * 100, 0.4, color=C["red"], label="Why-model (permutation importance)")
axes[0].set_yticks(yy); axes[0].set_yticklabels(top.index); axes[0].set_xlabel("Share of importance (%)")
axes[0].legend(frameon=False, loc="lower right"); axes[0].set_title(f"Predicting WHETHER vs WHY (Spearman ρ = {rho:.2f})")
panel(axes[0], "a")
ab = t12.copy(); ab["mean"] = ab["Macro AUC"].str.split(" ± ").str[0].astype(float); ab["sd"] = ab["Macro AUC"].str.split(" ± ").str[1].astype(float)
sets = ["All features", "Without Latitude/Longitude", "Geography only"]
for j, (mdl, colr) in enumerate([("Multinomial LogReg", C["dark"]), ("Gradient Boosting", C["red"])]):
    s_ = ab[ab.Model == mdl].set_index("Feature set").loc[sets]
    axes[1].bar(np.arange(3) + (j - 0.5) * 0.35, s_["mean"], 0.35, yerr=s_["sd"], color=colr, alpha=0.8, label=mdl, capsize=3)
axes[1].axhline(0.5, ls="--", color=C["grey"], lw=0.8); axes[1].set_ylim(0.45, 0.65)
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(["All", "No geography", "Geography only"])
axes[1].set_ylabel("Macro AUC (stated reason)"); axes[1].legend(frameon=False)
axes[1].set_title("Reason signal is almost entirely geographic"); panel(axes[1], "b")
savefig("Fig_7_whether_vs_why")

# ---- Fig 8: profit (recomputed from out-of-fold predictions, identical rules to NB5) ----
p_raw = SHAPM[PRIMARY]["prob"].values
pcal = np.zeros(N)
for k in np.unique(fold_of):
    pcal[fold_of == k] = IsotonicRegression(out_of_bounds="clip").fit(p_raw[fold_of != k], y[fold_of != k]).predict(p_raw[fold_of == k])
shap_dom = np.array(DOMS)[shares(SV[PRIMARY], MAPS["Primary"])[DOMS].values.argmax(1)]
maj_dom, why_dom = np.empty(N, object), np.empty(N, object)
for k in np.unique(fold_of):
    tr = (fold_of != k) & act; te = fold_of == k
    maj_dom[te] = pd.Series(true_dom[tr]).value_counts().index[0]
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    w = Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED))])
    why_dom[te] = w.fit(data.loc[tr, FEATS], true_dom[tr].astype(str)).predict(data.loc[te, FEATS])
unc, unk = rg == "Uncontrollable", rg == "Unknown"
def curve(score, od, gm=0.40, gx=0.10, cost=100):
    g = np.where(od == true_dom, gm, gx).astype(float); g[unk] = (gm + 2 * gx) / 3; g[unc] = 0; g[y == 0] = 0
    return np.concatenate([[0], np.cumsum((g * CLTV - cost)[np.argsort(-score, kind="stable")])])
sc = pcal * CLTV
curves = {"Majority offer (S2)": (curve(sc, maj_dom), C["blue"], "-"),
          "SHAP-guided offer (S3)": (curve(sc, shap_dom), C["red"], "-"),
          "Why-model offer (S4)": (curve(sc, why_dom), C["green"], "--"),
          "Oracle reason (S5, bound)": (curve(sc, true_dom), C["dark"], ":")}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1.2, 1]})
x = np.arange(N + 1) / N * 100
for lab, (cv, col, ls) in curves.items():
    axes[0].plot(x, cv / 1e6, color=col, ls=ls, lw=1.6, label=f"{lab}: max {cv.max()/1e6:.2f}M")
axes[0].axhline(0, color=C["grey"], lw=0.8); axes[0].set_xlim(0, 100)
axes[0].set_xlabel("% of customers targeted (ranked by calibrated risk × CLTV)"); axes[0].set_ylabel("Cumulative profit (millions)")
axes[0].legend(frameon=True, framealpha=0.9, edgecolor="none", loc="lower right"); axes[0].set_title("Profit curves (out-of-fold)"); panel(axes[0], "a")
sens = pd.read_csv(ff("nb5/tables/T18_sensitivity_SHAP-guided.csv"), index_col=0)
vmax = np.nanmax(np.abs(sens.values))
sns.heatmap(sens, annot=True, fmt=".0f", cmap="RdBu", center=0, vmin=-vmax, vmax=vmax, cbar_kws={"label": "% vs majority offer"},
            ax=axes[1], annot_kws={"size": 8})
axes[1].set_title("SHAP-guided offers lose profit in every setting" if (sens.values < 0).all() else "SHAP-guided offer: % profit vs majority")
axes[1].set_xlabel("Contact cost"); axes[1].set_ylabel("Offer success if matched")
panel(axes[1], "b")
savefig("Fig_8_profit")

# =====================================================================
# EXPORT: Excel workbook + captions
# =====================================================================
xlsx = os.path.join(OUT, "manuscript_tables.xlsx")
with pd.ExcelWriter(xlsx) as w:
    for key, (df, cap, idx) in TABLES.items():
        pd.DataFrame({cap: []}).to_excel(w, sheet_name=key, index=False)
        df.to_excel(w, sheet_name=key, startrow=2, index=idx)

S = lambda k: curves[k][0].max()
pct = lambda a, b: 100 * (a - b) / abs(b)
fp_real = FOREST[(FOREST.Map == "Primary")]
FIGCAP = {
 "Fig_1": "Reason-validated XAI framework: (1) leakage-free risk model, (2) explanation, (3) validation against stated reasons with a positive control, (4) why-model, (5) profit evaluation. [Draw in draw.io]",
 "Fig_2": f"Data overview. (a, b) Churn rate by contract and internet service (dashed: overall {overall:.1f}%). (c) Stated reason groups among {int((y==1).sum())} churners; grey groups are excluded from validation.",
 "Fig_3": f"Predictive performance under nested cross-validation. (a) AUC over {J} outer folds. (b) Critical-difference diagram; models within CD = {CD:.2f} are not significantly different. {fr_txt}.",
 "Fig_4": f"Explanations differ across near-equally accurate models. (a) Global SHAP importance. (b) Agreement of explanations between models: global Kendall τ (upper triangle) and mean customer-level top-3 Jaccard (lower triangle).",
 "Fig_5": f"Explanations do not reflect stated reasons. (a) Mean share of positive {PRIMARY} attribution per domain is nearly identical across reason groups. (b) Domain AUCs with 95% bootstrap CIs (range {fp_real.AUC.min():.3f}–{fp_real.AUC.max():.3f}); 0.5 = no information.",
 "Fig_6": "Positive control. Synthetic reasons with known feature dependence are detected for Support and Offering as signal increases; dotted lines show real reasons. (b) Robustness map with Total Charges reassigned to the Relationship domain.",
 "Fig_7": f"Risk and reasons rely on different information. (a) Feature importance of the churn model vs the why-model (Spearman ρ = {rho:.2f}). (b) Recoverability of stated reasons collapses to chance without geography.",
 "Fig_8": f"Retention profit. (a) Profit curves: SHAP-guided offers {pct(S('SHAP-guided offer (S3)'), S('Majority offer (S2)')):+.1f}% and oracle reasons {pct(S('Oracle reason (S5, bound)'), S('Majority offer (S2)')):+.1f}% relative to the majority offer. (b) Sensitivity of the SHAP-guided loss to offer success and contact cost.",
}
with open(os.path.join(OUT, "captions.md"), "w") as f:
    f.write("# Draft captions (numbers filled from results)\n\n## Figures\n")
    for k, v in FIGCAP.items(): f.write(f"**{k.replace('_', ' ')}.** {v}\n\n")
    f.write("## Tables\n")
    for k, v in CAPTIONS.items(): f.write(f"**{k.replace('_', ' ')}.** {v}\n\n")

print("\n" + "=" * 70)
print("FINAL OUTPUT ->", OUT)
print("Figures:", sorted(f for f in os.listdir(FIG) if f.endswith(".png")))
print("Tables :", sorted(os.listdir(TAB)))
print("Excel  :", xlsx)
print("Captions: captions.md")
print("\nROBUSTNESS (Total Charges -> Relationship): see Table_6 rows 'Robustness' and Fig_6(b).")
print("Final tables and figures saved in", os.path.join(WORK_DIR, "final"))

## Step 07: Extension 1: counterfactual explanations and power analysis


In [ ]:
# =====================================================================
# NB7 (single cell): Counterfactual explanations vs stated reasons
#                    + statistical power of the validation
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# Why: prior work prescribes retention actions with counterfactuals
#      (DiCE-style). Reviewers will ask whether the null result holds
#      for counterfactuals, not only SHAP/LIME.
#   A. Minimal-change counterfactuals: for every validation churner, the
#      smallest set (1 or 2) of ACTIONABLE feature changes that moves the
#      out-of-fold churn model below the decision threshold (exhaustive
#      search -> deterministic, reproducible)
#   B. Do counterfactual domains match stated reasons? (same metrics as NB3)
#   C. Profit of counterfactual-guided offers (same engine as NB5)
#   D. Power: minimum detectable domain AUC for the validation sample
#
# Input : NB1-NB5 output in /kaggle/working     RUNTIME: ~5-10 min
# Output: /kaggle/working/nb7/
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

THRESH  = 0.5      # counterfactual must bring raw churn probability below this
N_PERM  = 2000
N_BOOT  = 1000
SEED    = 42
G_MATCH, G_MISS, COST = 0.40, 0.10, 100

import os, json, glob, itertools, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, cohen_kappa_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

HAS = {}
try:
    from catboost import CatBoostClassifier; HAS["CatBoost"] = True
except ImportError: pass
try:
    from xgboost import XGBClassifier; HAS["XGBoost"] = True
except ImportError: pass

BASE = os.environ.get("NB_IN", WORK_DIR)
def ff(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found - run NB1-NB5 first")
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb7")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "pdf.fonttype": 42})

cfg    = json.load(open(ff("config.json")))
data   = pd.read_csv(ff("telco_clean.csv"))
splits = [s for s in json.load(open(ff("nb2/outer_splits.json"))) if s["repeat"] == 0]
bparam = pd.read_csv(ff("nb2/best_params.csv"))
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT
X, y = data[FEATS].copy(), data[cfg["target"]].astype(int).values
CLTV, N = data[cfg["value_col"]].astype(float).values, len(data)
fold_of = np.empty(N, int)
for s in splits: fold_of[s["test_idx"]] = s["fold"]

PRIMARY = next(m for m in ["CatBoost", "XGBoost", "LogReg"] if (m == "LogReg" or m in HAS) and (bparam.Model == m).any())
print("Counterfactuals generated for:", PRIMARY)

# ---------------------------------------------------------------------
# Reason ground truth + domains (identical to NB3/NB6 primary map)
# ---------------------------------------------------------------------
R2D = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
DOMS = ["Price", "Support", "Offering"]
FEAT_DOMAIN = {**{f: "Price" for f in ["Monthly Charges", "Total Charges", "Charge Gap"]},
               **{f: "Support" for f in ["Tech Support", "Online Security", "Device Protection"]},
               **{f: "Offering" for f in ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines",
                                          "Online Backup", "Phone Service", "Num Services"]},
               **{f: "Relationship" for f in ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"]}}
rg = data["Reason Group"].astype(object).values
true_dom = np.array([R2D.get(r) if isinstance(r, str) else None for r in rg], dtype=object)
act = (y == 1) & np.isin(rg, list(R2D))
val_idx = np.where(act)[0]
print(f"Validation churners: {len(val_idx)}")

# ---------------------------------------------------------------------
# Refit out-of-fold primary model (NB2 tuned params, as in NB3)
# ---------------------------------------------------------------------
INT_P = {"n_estimators", "max_depth", "min_samples_leaf", "num_leaves", "min_child_samples", "min_child_weight", "iterations", "depth"}
def get_params(m, k):
    r = bparam[(bparam.Model == m) & (bparam.Fold == k)].iloc[0].to_dict()
    return {kk: (int(v) if kk in INT_P else v) for kk, v in r.items() if kk not in ("Model", "Fold") and not (isinstance(v, float) and np.isnan(v))}
def build(m, p, seed, pos_w):
    if m == "CatBoost":
        return CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=float(p["learning_rate"]),
                                  l2_leaf_reg=float(p["l2_leaf_reg"]), auto_class_weights="Balanced", random_seed=seed, verbose=0, thread_count=-1)
    if m == "XGBoost":
        return XGBClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"], learning_rate=float(p["learning_rate"]),
                             subsample=float(p["subsample"]), colsample_bytree=float(p["colsample_bytree"]), min_child_weight=p["min_child_weight"],
                             reg_lambda=float(p["reg_lambda"]), scale_pos_weight=pos_w, eval_metric="logloss", tree_method="hist",
                             n_jobs=-1, random_state=seed, verbosity=0)
    return LogisticRegression(C=float(p["C"]), class_weight="balanced", max_iter=3000)
def pipe_for(k):
    tr = fold_of != k
    pos_w = (y[tr] == 0).sum() / (y[tr] == 1).sum()
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    return Pipeline([("pre", pre), ("clf", build(PRIMARY, get_params(PRIMARY, k), SEED + k, pos_w))]).fit(X[tr], y[tr])

# ---------------------------------------------------------------------
# A. Actionable changes (immutable: tenure, demographics, location, history)
# ---------------------------------------------------------------------
ADDONS = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]
CONTRACTS = sorted(X["Contract"].unique()); PAYMENTS = sorted(X["Payment Method"].unique())
def candidate_changes(row):
    """List of (feature, new_value) single changes that are feasible for this customer."""
    ch = [("Contract", v) for v in CONTRACTS if v != row["Contract"]]
    ch += [("Payment Method", v) for v in PAYMENTS if v != row["Payment Method"]]
    ch += [("Paperless Billing", 1 - row["Paperless Billing"])]
    if row["Internet Service"] != "No":
        ch += [(a, "Yes" if row[a] == "No" else "No") for a in ADDONS]
        ch += [("Internet Service", v) for v in ["DSL", "Fiber optic"] if v != row["Internet Service"]]
    if row["Phone Service"] == 1:
        ch += [("Multiple Lines", "Yes" if row["Multiple Lines"] == "No" else "No")]
    ch += [("Monthly Charges", f) for f in (0.9, 0.8, 0.7)]          # price reduction of 10/20/30%
    return ch
def apply(row, changes):
    r = row.copy()
    for f, v in changes:
        if f == "Monthly Charges":
            delta = r["Monthly Charges"] * (1 - v)
            r["Monthly Charges"] -= delta; r["Charge Gap"] -= delta
        else:
            r[f] = v
    r["Num Services"] = sum(r[a] == "Yes" for a in ADDONS) + r["Phone Service"] + int(r["Multiple Lines"] == "Yes")
    return r

t0 = time.time(); CF = {}
for k in np.unique(fold_of):
    pipe = pipe_for(k)
    ids = val_idx[fold_of[val_idx] == k]
    rows, meta = [], []
    for i in ids:
        row = X.iloc[i]; ch = candidate_changes(row)
        combos = [(c,) for c in ch] + [c for c in itertools.combinations(ch, 2) if c[0][0] != c[1][0]]
        for c in combos:
            rows.append(apply(row, c)); meta.append((i, c))
    P = pipe.predict_proba(pd.DataFrame(rows)[FEATS])[:, 1]
    p0 = pipe.predict_proba(X.iloc[ids])[:, 1]
    base = dict(zip(ids, p0))
    df = pd.DataFrame({"i": [m[0] for m in meta], "changes": [m[1] for m in meta], "p": P})
    df["n"] = df["changes"].map(len); df["flip"] = df["p"] < THRESH
    for i, g in df.groupby("i"):
        flips = g[g.flip]
        pick = (flips.sort_values(["n", "p"]).iloc[0] if len(flips) else g.sort_values(["p", "n"]).iloc[0])
        CF[i] = {"changes": pick.changes, "p_before": base[i], "p_after": pick.p, "flipped": bool(len(flips)), "n_changes": pick.n}
    print(f"fold {k+1}: {len(ids)} churners, {len(df):,} candidates | {(time.time()-t0)/60:.1f} min")

def cf_domains(changes):
    return [FEAT_DOMAIN[f] for f, _ in changes]
cf = pd.DataFrame.from_dict(CF, orient="index").loc[val_idx]
cf["domains"] = cf["changes"].map(cf_domains)
cf["features"] = cf["changes"].map(lambda c: " + ".join(f for f, _ in c))
cf["indicated"] = cf["domains"].map(lambda ds: next((d for d in ds if d in DOMS), "Relationship"))
cf["true"] = true_dom[val_idx]
cf.drop(columns=["changes"]).assign(domains=cf["domains"].map("|".join)).to_csv(os.path.join(OUT, "counterfactuals.csv"))

summ = pd.Series({
    "Churners": len(cf), "Flipped below threshold (%)": 100 * cf.flipped.mean(),
    "Single-change CFs (%)": 100 * (cf.n_changes == 1).mean(),
    "Mean churn prob. before": cf.p_before.mean(), "Mean churn prob. after": cf.p_after.mean(),
    "CFs touching only Relationship features (%)": 100 * (cf.indicated == "Relationship").mean()}).round(3)
top_cf = cf["features"].value_counts().head(10).rename("Customers").to_frame()
top_cf["%"] = (100 * top_cf["Customers"] / len(cf)).round(1)
summ.to_csv(os.path.join(TAB, "T20_cf_summary.csv")); top_cf.to_csv(os.path.join(TAB, "T21_cf_top_changes.csv"))
print("\n=== Counterfactual summary ==="); display(summ.to_frame("Value"))
print("\n=== Most frequent minimal counterfactuals ==="); display(top_cf)

# ---------------------------------------------------------------------
# B. Do counterfactual domains match stated reasons?
# ---------------------------------------------------------------------
td, ind = cf["true"].values, cf["indicated"].values
acc = (ind == td).mean()
perm = np.array([(ind == rng.permutation(td)).mean() for _ in range(N_PERM)])
row = {"Explanation": f"Counterfactual-{PRIMARY}", "Alignment acc.": round(acc, 3),
       "Permutation p": round((np.sum(perm >= acc) + 1) / (N_PERM + 1), 3),
       "Majority baseline": round(pd.Series(td).value_counts(normalize=True).iloc[0], 3),
       "Cohen κ": round(cohen_kappa_score(td, ind), 3)}
for d in DOMS:
    lab = (td == d).astype(int)
    sc = cf["domains"].map(lambda ds: ds.count(d) / len(ds)).values
    auc = roc_auc_score(lab, sc)
    bs = [roc_auc_score(lab[b], sc[b]) for b in (rng.integers(0, len(lab), len(lab)) for _ in range(N_BOOT)) if lab[b].min() != lab[b].max()]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    row[f"AUC {d} [95% CI]"] = f"{auc:.3f} [{lo:.3f}, {hi:.3f}]"
row["Relationship-only share"] = round((ind == "Relationship").mean(), 3)
cf_val = pd.DataFrame([row]).set_index("Explanation")
cf_val.to_csv(os.path.join(TAB, "T22_cf_reason_validation.csv"))
print("\n=== Counterfactual explanations vs stated reasons ==="); display(cf_val)
ct = pd.crosstab(pd.Series(td, name="Stated domain"), pd.Series(ind, name="Counterfactual domain"), normalize="index").mul(100).round(1)
ct.to_csv(os.path.join(TAB, "T23_cf_confusion.csv")); display(ct)

# ---------------------------------------------------------------------
# C. Profit of counterfactual-guided offers (NB5 engine, out-of-fold)
# ---------------------------------------------------------------------
p_raw = pd.read_csv(ff(f"nb3/shap_{PRIMARY}.csv"))["prob"].values
pcal = np.zeros(N)
for k in np.unique(fold_of):
    pcal[fold_of == k] = IsotonicRegression(out_of_bounds="clip").fit(p_raw[fold_of != k], y[fold_of != k]).predict(p_raw[fold_of == k])
maj = np.empty(N, object)
for k in np.unique(fold_of):
    maj[fold_of == k] = pd.Series(true_dom[(fold_of != k) & act]).value_counts().index[0]
unc, unk = rg == "Uncontrollable", rg == "Unknown"
def curve(od, idx=None, gm=G_MATCH, gx=G_MISS, cost=COST):
    idx = np.arange(N) if idx is None else idx
    g = np.where(od == true_dom, gm, gx).astype(float); g[unk] = (gm + 2 * gx) / 3; g[unc] = 0; g[y == 0] = 0
    sc = (pcal * CLTV)[idx]
    return np.concatenate([[0], np.cumsum((g * CLTV - cost)[idx][np.argsort(-sc, kind="stable")])]).max()
cf_contract = maj.copy(); cf_contract[val_idx] = cf["indicated"].values          # Relationship -> retention-contract offer (matches no reason)
cf_fallback = cf_contract.copy(); rel = val_idx[cf["indicated"].values == "Relationship"]; cf_fallback[rel] = maj[rel]
prow = []
for name, od in [("S2 Majority offer", maj), ("S6 Counterfactual offer", cf_contract), ("S6f Counterfactual offer, majority fallback", cf_fallback)]:
    mp = curve(od); diffs = []
    for _ in range(N_BOOT):
        b = rng.integers(0, N, N); diffs.append(curve(od, b) - curve(maj, b))
    prow.append({"Strategy": name, "Max profit": round(mp), "% vs S2": round(100 * (mp - curve(maj)) / curve(maj), 1),
                 "Δ vs S2 [95% CI]": "—" if name.startswith("S2") else f"{np.mean(diffs):,.0f} [{np.percentile(diffs, 2.5):,.0f}, {np.percentile(diffs, 97.5):,.0f}]",
                 "Offer–reason match": round((od[val_idx] == true_dom[val_idx]).mean(), 3)})
cf_profit = pd.DataFrame(prow).set_index("Strategy")
cf_profit.to_csv(os.path.join(TAB, "T24_cf_profit.csv"))
print("\n=== Profit of counterfactual-guided offers ==="); display(cf_profit)

# ---------------------------------------------------------------------
# D. Power: minimum detectable domain AUC (Hanley & McNeil 1982), alpha=.05 two-sided, power .80
# ---------------------------------------------------------------------
def se_auc(A, n1, n0):
    Q1, Q2 = A / (2 - A), 2 * A * A / (1 + A)
    return np.sqrt((A * (1 - A) + (n1 - 1) * (Q1 - A * A) + (n0 - 1) * (Q2 - A * A)) / (n1 * n0))
pw = []
for d in DOMS:
    n1 = int((td == d).sum()); n0 = len(td) - n1
    grid = np.arange(0.5005, 0.7, 0.0005)
    mde = next(A for A in grid if (A - 0.5) >= (1.96 * se_auc(0.5, n1, n0) + 0.8416 * se_auc(A, n1, n0)))
    pw.append({"Domain": d, "Churners stating it": n1, "Others": n0, "Minimum detectable AUC (80% power)": round(mde, 3)})
power = pd.DataFrame(pw).set_index("Domain")
power.to_csv(os.path.join(TAB, "T25_power.csv"))
print("\n=== Power of the domain-AUC test ==="); display(power)

# ---------------------------------------------------------------------
# Figure: where counterfactuals point vs what customers said
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 3.2))
cols = ["Price", "Support", "Offering", "Relationship"]
ct2 = ct.reindex(columns=cols, fill_value=0)
left = np.zeros(len(ct2))
pal = {"Price": "#C0392B", "Support": "#3B5B92", "Offering": "#4E8A55", "Relationship": "#8C939D"}
for c in cols:
    ax.barh(ct2.index, ct2[c], left=left, color=pal[c], label=c, height=0.6); left += ct2[c].values
ax.set_xlabel("% of churners"); ax.set_ylabel("Stated reason domain"); ax.set_xlim(0, 100)
ax.legend(title="Counterfactual changes", frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=8)
plt.tight_layout()
for ext in ["png", "pdf"]: plt.savefig(os.path.join(FIG, f"Fig_9_counterfactual_domains.{ext}"), bbox_inches="tight")
plt.show()
print("\nSaved to", OUT, "->", sorted(os.listdir(TAB)))

## Step 07b: Extension 1 (robustness): restricted counterfactuals


In [ ]:
# =====================================================================
# NB7b (single cell): ROBUSTNESS - counterfactuals WITHOUT contract/payment/paperless changes
# (answers: 'counterfactuals point to Contract only because the search favours it')
# NB7 (base): Counterfactual explanations vs stated reasons
#                    + statistical power of the validation
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# Why: prior work prescribes retention actions with counterfactuals
#      (DiCE-style). Reviewers will ask whether the null result holds
#      for counterfactuals, not only SHAP/LIME.
#   A. Minimal-change counterfactuals: for every validation churner, the
#      smallest set (1 or 2) of ACTIONABLE feature changes that moves the
#      out-of-fold churn model below the decision threshold (exhaustive
#      search -> deterministic, reproducible)
#   B. Do counterfactual domains match stated reasons? (same metrics as NB3)
#   C. Profit of counterfactual-guided offers (same engine as NB5)
#   D. Power: minimum detectable domain AUC for the validation sample
#
# Input : NB1-NB5 output in /kaggle/working     RUNTIME: ~5-10 min
# Output: /kaggle/working/nb7/
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

EXCLUDE = {"Contract", "Payment Method", "Paperless Billing"}   # Relationship levers removed
THRESH  = 0.5      # counterfactual must bring raw churn probability below this
N_PERM  = 2000
N_BOOT  = 1000
SEED    = 42
G_MATCH, G_MISS, COST = 0.40, 0.10, 100

import os, json, glob, itertools, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, cohen_kappa_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

HAS = {}
try:
    from catboost import CatBoostClassifier; HAS["CatBoost"] = True
except ImportError: pass
try:
    from xgboost import XGBClassifier; HAS["XGBoost"] = True
except ImportError: pass

BASE = os.environ.get("NB_IN", WORK_DIR)
def ff(name):
    for p in [os.path.join(BASE, name)] + glob.glob(f"{DATA_DIR}/**/{name}", recursive=True):
        if os.path.exists(p): return p
    raise FileNotFoundError(f"{name} not found - run NB1-NB5 first")
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb7b")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "pdf.fonttype": 42})

cfg    = json.load(open(ff("config.json")))
data   = pd.read_csv(ff("telco_clean.csv"))
splits = [s for s in json.load(open(ff("nb2/outer_splits.json"))) if s["repeat"] == 0]
bparam = pd.read_csv(ff("nb2/best_params.csv"))
NUM = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"]
BIN, CAT = cfg["bin_features"], cfg["cat_features"]
FEATS = NUM + BIN + CAT
X, y = data[FEATS].copy(), data[cfg["target"]].astype(int).values
CLTV, N = data[cfg["value_col"]].astype(float).values, len(data)
fold_of = np.empty(N, int)
for s in splits: fold_of[s["test_idx"]] = s["fold"]

PRIMARY = next(m for m in ["CatBoost", "XGBoost", "LogReg"] if (m == "LogReg" or m in HAS) and (bparam.Model == m).any())
print("Counterfactuals generated for:", PRIMARY)

# ---------------------------------------------------------------------
# Reason ground truth + domains (identical to NB3/NB6 primary map)
# ---------------------------------------------------------------------
R2D = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
DOMS = ["Price", "Support", "Offering"]
FEAT_DOMAIN = {**{f: "Price" for f in ["Monthly Charges", "Total Charges", "Charge Gap"]},
               **{f: "Support" for f in ["Tech Support", "Online Security", "Device Protection"]},
               **{f: "Offering" for f in ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines",
                                          "Online Backup", "Phone Service", "Num Services"]},
               **{f: "Relationship" for f in ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"]}}
rg = data["Reason Group"].astype(object).values
true_dom = np.array([R2D.get(r) if isinstance(r, str) else None for r in rg], dtype=object)
act = (y == 1) & np.isin(rg, list(R2D))
val_idx = np.where(act)[0]
print(f"Validation churners: {len(val_idx)}")

# ---------------------------------------------------------------------
# Refit out-of-fold primary model (NB2 tuned params, as in NB3)
# ---------------------------------------------------------------------
INT_P = {"n_estimators", "max_depth", "min_samples_leaf", "num_leaves", "min_child_samples", "min_child_weight", "iterations", "depth"}
def get_params(m, k):
    r = bparam[(bparam.Model == m) & (bparam.Fold == k)].iloc[0].to_dict()
    return {kk: (int(v) if kk in INT_P else v) for kk, v in r.items() if kk not in ("Model", "Fold") and not (isinstance(v, float) and np.isnan(v))}
def build(m, p, seed, pos_w):
    if m == "CatBoost":
        return CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=float(p["learning_rate"]),
                                  l2_leaf_reg=float(p["l2_leaf_reg"]), auto_class_weights="Balanced", random_seed=seed, verbose=0, thread_count=-1)
    if m == "XGBoost":
        return XGBClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"], learning_rate=float(p["learning_rate"]),
                             subsample=float(p["subsample"]), colsample_bytree=float(p["colsample_bytree"]), min_child_weight=p["min_child_weight"],
                             reg_lambda=float(p["reg_lambda"]), scale_pos_weight=pos_w, eval_metric="logloss", tree_method="hist",
                             n_jobs=-1, random_state=seed, verbosity=0)
    return LogisticRegression(C=float(p["C"]), class_weight="balanced", max_iter=3000)
def pipe_for(k):
    tr = fold_of != k
    pos_w = (y[tr] == 0).sum() / (y[tr] == 1).sum()
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("bin", "passthrough", BIN),
                             ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])
    return Pipeline([("pre", pre), ("clf", build(PRIMARY, get_params(PRIMARY, k), SEED + k, pos_w))]).fit(X[tr], y[tr])

# ---------------------------------------------------------------------
# A. Actionable changes (immutable: tenure, demographics, location, history)
# ---------------------------------------------------------------------
ADDONS = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]
CONTRACTS = sorted(X["Contract"].unique()); PAYMENTS = sorted(X["Payment Method"].unique())
def candidate_changes(row):
    """List of (feature, new_value) single changes that are feasible for this customer."""
    ch = [("Contract", v) for v in CONTRACTS if v != row["Contract"]]
    ch += [("Payment Method", v) for v in PAYMENTS if v != row["Payment Method"]]
    ch += [("Paperless Billing", 1 - row["Paperless Billing"])]
    if row["Internet Service"] != "No":
        ch += [(a, "Yes" if row[a] == "No" else "No") for a in ADDONS]
        ch += [("Internet Service", v) for v in ["DSL", "Fiber optic"] if v != row["Internet Service"]]
    if row["Phone Service"] == 1:
        ch += [("Multiple Lines", "Yes" if row["Multiple Lines"] == "No" else "No")]
    ch += [("Monthly Charges", f) for f in (0.9, 0.8, 0.7)]          # price reduction of 10/20/30%
    return [c for c in ch if c[0] not in EXCLUDE]
def apply(row, changes):
    r = row.copy()
    for f, v in changes:
        if f == "Monthly Charges":
            delta = r["Monthly Charges"] * (1 - v)
            r["Monthly Charges"] -= delta; r["Charge Gap"] -= delta
        else:
            r[f] = v
    r["Num Services"] = sum(r[a] == "Yes" for a in ADDONS) + r["Phone Service"] + int(r["Multiple Lines"] == "Yes")
    return r

t0 = time.time(); CF = {}
for k in np.unique(fold_of):
    pipe = pipe_for(k)
    ids = val_idx[fold_of[val_idx] == k]
    rows, meta = [], []
    for i in ids:
        row = X.iloc[i]; ch = candidate_changes(row)
        combos = [(c,) for c in ch] + [c for c in itertools.combinations(ch, 2) if c[0][0] != c[1][0]]
        for c in combos:
            rows.append(apply(row, c)); meta.append((i, c))
    P = pipe.predict_proba(pd.DataFrame(rows)[FEATS])[:, 1]
    p0 = pipe.predict_proba(X.iloc[ids])[:, 1]
    base = dict(zip(ids, p0))
    df = pd.DataFrame({"i": [m[0] for m in meta], "changes": [m[1] for m in meta], "p": P})
    df["n"] = df["changes"].map(len); df["flip"] = df["p"] < THRESH
    for i, g in df.groupby("i"):
        flips = g[g.flip]
        pick = (flips.sort_values(["n", "p"]).iloc[0] if len(flips) else g.sort_values(["p", "n"]).iloc[0])
        CF[i] = {"changes": pick.changes, "p_before": base[i], "p_after": pick.p, "flipped": bool(len(flips)), "n_changes": pick.n}
    print(f"fold {k+1}: {len(ids)} churners, {len(df):,} candidates | {(time.time()-t0)/60:.1f} min")

def cf_domains(changes):
    return [FEAT_DOMAIN[f] for f, _ in changes]
cf = pd.DataFrame.from_dict(CF, orient="index").loc[val_idx]
cf["domains"] = cf["changes"].map(cf_domains)
cf["features"] = cf["changes"].map(lambda c: " + ".join(f for f, _ in c))
cf["indicated"] = cf["domains"].map(lambda ds: next((d for d in ds if d in DOMS), "Relationship"))
cf["true"] = true_dom[val_idx]
cf.drop(columns=["changes"]).assign(domains=cf["domains"].map("|".join)).to_csv(os.path.join(OUT, "counterfactuals.csv"))

summ = pd.Series({
    "Churners": len(cf), "Flipped below threshold (%)": 100 * cf.flipped.mean(),
    "Single-change CFs (%)": 100 * (cf.n_changes == 1).mean(),
    "Mean churn prob. before": cf.p_before.mean(), "Mean churn prob. after": cf.p_after.mean(),
    "CFs touching only Relationship features (%)": 100 * (cf.indicated == "Relationship").mean()}).round(3)
top_cf = cf["features"].value_counts().head(10).rename("Customers").to_frame()
top_cf["%"] = (100 * top_cf["Customers"] / len(cf)).round(1)
summ.to_csv(os.path.join(TAB, "T20_cf_summary.csv")); top_cf.to_csv(os.path.join(TAB, "T21_cf_top_changes.csv"))
print("\n=== Counterfactual summary ==="); display(summ.to_frame("Value"))
print("\n=== Most frequent minimal counterfactuals ==="); display(top_cf)

# ---------------------------------------------------------------------
# B. Do counterfactual domains match stated reasons?
# ---------------------------------------------------------------------
td, ind = cf["true"].values, cf["indicated"].values
acc = (ind == td).mean()
perm = np.array([(ind == rng.permutation(td)).mean() for _ in range(N_PERM)])
row = {"Explanation": f"Counterfactual-{PRIMARY}", "Alignment acc.": round(acc, 3),
       "Permutation p": round((np.sum(perm >= acc) + 1) / (N_PERM + 1), 3),
       "Majority baseline": round(pd.Series(td).value_counts(normalize=True).iloc[0], 3),
       "Cohen κ": round(cohen_kappa_score(td, ind), 3)}
for d in DOMS:
    lab = (td == d).astype(int)
    sc = cf["domains"].map(lambda ds: ds.count(d) / len(ds)).values
    auc = roc_auc_score(lab, sc)
    bs = [roc_auc_score(lab[b], sc[b]) for b in (rng.integers(0, len(lab), len(lab)) for _ in range(N_BOOT)) if lab[b].min() != lab[b].max()]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    row[f"AUC {d} [95% CI]"] = f"{auc:.3f} [{lo:.3f}, {hi:.3f}]"
row["Relationship-only share"] = round((ind == "Relationship").mean(), 3)
cf_val = pd.DataFrame([row]).set_index("Explanation")
cf_val.to_csv(os.path.join(TAB, "T22_cf_reason_validation.csv"))
print("\n=== Counterfactual explanations vs stated reasons ==="); display(cf_val)
ct = pd.crosstab(pd.Series(td, name="Stated domain"), pd.Series(ind, name="Counterfactual domain"), normalize="index").mul(100).round(1)
ct.to_csv(os.path.join(TAB, "T23_cf_confusion.csv")); display(ct)

# ---------------------------------------------------------------------
# C. Profit of counterfactual-guided offers (NB5 engine, out-of-fold)
# ---------------------------------------------------------------------
p_raw = pd.read_csv(ff(f"nb3/shap_{PRIMARY}.csv"))["prob"].values
pcal = np.zeros(N)
for k in np.unique(fold_of):
    pcal[fold_of == k] = IsotonicRegression(out_of_bounds="clip").fit(p_raw[fold_of != k], y[fold_of != k]).predict(p_raw[fold_of == k])
maj = np.empty(N, object)
for k in np.unique(fold_of):
    maj[fold_of == k] = pd.Series(true_dom[(fold_of != k) & act]).value_counts().index[0]
unc, unk = rg == "Uncontrollable", rg == "Unknown"
def curve(od, idx=None, gm=G_MATCH, gx=G_MISS, cost=COST):
    idx = np.arange(N) if idx is None else idx
    g = np.where(od == true_dom, gm, gx).astype(float); g[unk] = (gm + 2 * gx) / 3; g[unc] = 0; g[y == 0] = 0
    sc = (pcal * CLTV)[idx]
    return np.concatenate([[0], np.cumsum((g * CLTV - cost)[idx][np.argsort(-sc, kind="stable")])]).max()
cf_contract = maj.copy(); cf_contract[val_idx] = cf["indicated"].values          # Relationship -> retention-contract offer (matches no reason)
cf_fallback = cf_contract.copy(); rel = val_idx[cf["indicated"].values == "Relationship"]; cf_fallback[rel] = maj[rel]
prow = []
for name, od in [("S2 Majority offer", maj), ("S6r Restricted counterfactual offer", cf_contract), ("S6rf Restricted counterfactual, majority fallback", cf_fallback)]:
    mp = curve(od); diffs = []
    for _ in range(N_BOOT):
        b = rng.integers(0, N, N); diffs.append(curve(od, b) - curve(maj, b))
    prow.append({"Strategy": name, "Max profit": round(mp), "% vs S2": round(100 * (mp - curve(maj)) / curve(maj), 1),
                 "Δ vs S2 [95% CI]": "—" if name.startswith("S2") else f"{np.mean(diffs):,.0f} [{np.percentile(diffs, 2.5):,.0f}, {np.percentile(diffs, 97.5):,.0f}]",
                 "Offer–reason match": round((od[val_idx] == true_dom[val_idx]).mean(), 3)})
cf_profit = pd.DataFrame(prow).set_index("Strategy")
cf_profit.to_csv(os.path.join(TAB, "T24_cf_profit.csv"))
print("\n=== Profit of counterfactual-guided offers ==="); display(cf_profit)

# ---------------------------------------------------------------------
# D. Power: minimum detectable domain AUC (Hanley & McNeil 1982), alpha=.05 two-sided, power .80
# ---------------------------------------------------------------------
def se_auc(A, n1, n0):
    Q1, Q2 = A / (2 - A), 2 * A * A / (1 + A)
    return np.sqrt((A * (1 - A) + (n1 - 1) * (Q1 - A * A) + (n0 - 1) * (Q2 - A * A)) / (n1 * n0))
pw = []
for d in DOMS:
    n1 = int((td == d).sum()); n0 = len(td) - n1
    grid = np.arange(0.5005, 0.7, 0.0005)
    mde = next(A for A in grid if (A - 0.5) >= (1.96 * se_auc(0.5, n1, n0) + 0.8416 * se_auc(A, n1, n0)))
    pw.append({"Domain": d, "Churners stating it": n1, "Others": n0, "Minimum detectable AUC (80% power)": round(mde, 3)})
power = pd.DataFrame(pw).set_index("Domain")
power.to_csv(os.path.join(TAB, "T25_power.csv"))
print("\n=== Power of the domain-AUC test ==="); display(power)

# ---------------------------------------------------------------------
# Figure: where counterfactuals point vs what customers said
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.2, 3.2))
cols = ["Price", "Support", "Offering", "Relationship"]
ct2 = ct.reindex(columns=cols, fill_value=0)
left = np.zeros(len(ct2))
pal = {"Price": "#C0392B", "Support": "#3B5B92", "Offering": "#4E8A55", "Relationship": "#8C939D"}
for c in cols:
    ax.barh(ct2.index, ct2[c], left=left, color=pal[c], label=c, height=0.6); left += ct2[c].values
ax.set_xlabel("% of churners"); ax.set_ylabel("Stated reason domain"); ax.set_xlim(0, 100)
ax.legend(title="Counterfactual changes", frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.02), fontsize=8)
plt.tight_layout()
for ext in ["png", "pdf"]: plt.savefig(os.path.join(FIG, f"Fig_S1_counterfactual_domains_restricted.{ext}"), bbox_inches="tight")
plt.show()
print("\nSaved to", OUT, "->", sorted(os.listdir(TAB)))

## Step 08: Extension 2: Study 2 on richer customer records


In [ ]:
# =====================================================================
# NB8 (single cell): STUDY 2 — Does richer CRM data close the gap?
# Data: IBM Telco Customer Churn 11.1.3+ FULL release (Kaggle:
#       alfathterry/telco-customer-churn-11-1-3). Same customers as Study 1,
#       but with billing and usage features that map DIRECTLY onto stated
#       reasons (extra data charges, long-distance charges, download volume,
#       unlimited data, offers, refunds).
# Tests:
#   A. Leakage audit incl. Satisfaction Score / Customer Status
#   B. Risk model (CatBoost + LogReg, stratified 5-fold, out-of-fold)
#   C. Domain-level reason validation (same protocol as Study 1)
#   D. Feature-level test: for reasons with a DIRECTLY matching feature,
#      (i) do these churners differ on that feature?  (ii) does SHAP
#      single out that feature for them?
#   E. Reason recoverability (why-model) with/without geography
# RUNTIME: ~5-10 min.  Output: /kaggle/working/nb8/
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

SEED, N_PERM, N_BOOT, K = 42, 2000, 1000, 5

import os, re, glob, json, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, cohen_kappa_score, average_precision_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)
try:
    from catboost import CatBoostClassifier, Pool; HAS_CB = True
except ImportError:
    HAS_CB = False; print("catboost not found -> LogReg only")

OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb8"); TAB = os.path.join(OUT, "tables")
os.makedirs(TAB, exist_ok=True)
def save(df, name, index=True): df.to_csv(os.path.join(TAB, name + ".csv"), index=index); display(df)

# ---------------------------------------------------------------------
# 0. Load: largest csv/xlsx in an attached dataset whose path mentions 11-1-3 / 11.1.3
# ---------------------------------------------------------------------
DATA_PATH = ""   # optional: paste the exact file path here if auto-detection fails
# Auto-detect by COLUMNS (not by folder name): the 11.1.3+ release is the only one
# with billing columns such as "Total Extra Data Charges" / "Customer Status".
MARKERS = {"totalextradatacharges", "customerstatus", "avgmonthlygbdownload", "totallongdistancecharges", "churncategory"}
def header(p):
    try:
        h = pd.read_excel(p, nrows=0) if p.lower().endswith((".xlsx", ".xls")) else pd.read_csv(p, nrows=0)
        return {re.sub(r"[^a-z0-9]", "", str(c).lower()) for c in h.columns}
    except Exception: return set()
files = [p for p in glob.glob(os.environ.get("NB8_GLOB", DATA_DIR + "/**/*"), recursive=True)
         if p.lower().endswith((".csv", ".xlsx", ".xls"))]
print(f"{len(files)} csv/xlsx files under {DATA_DIR}:"); [print("  ", p) for p in files]
if DATA_PATH:
    path = DATA_PATH
else:
    score = {p: len(header(p) & MARKERS) for p in files}
    best = [p for p, s in score.items() if s >= 2]
    assert best, ("No file with IBM 11.1.3+ columns found. In the right panel: + Add Input -> search "
                  "'telco customer churn 11.1.3' -> Add. Or set DATA_PATH to the file path printed above.")
    path = max(best, key=lambda p: (score[p], os.path.getsize(p)))
raw = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
print("Loaded:", path, raw.shape)

norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
COL = {norm(c): c for c in raw.columns}
def col(*names, required=True):
    for n in names:
        if norm(n) in COL: return COL[norm(n)]
    if required: raise KeyError(f"None of {names} found. Columns: {list(raw.columns)}")
    return None

# Roles (first matching name wins). Missing optional columns are skipped.
C = dict(
    id=col("Customer ID", "CustomerID"), status=col("Customer Status", required=False),
    churn=col("Churn Label", "Churn", "Churn Value"), reason=col("Churn Reason"),
    category=col("Churn Category", required=False), score=col("Churn Score", required=False),
    satisfaction=col("Satisfaction Score", required=False), cltv=col("CLTV", required=False))
y_raw = raw[C["churn"]]
y = (y_raw.astype(str).str.lower().isin(["yes", "1", "true", "churned"])).astype(int).values
print(f"Churn rate: {y.mean():.4f} | churners: {y.sum()}")

DOMAIN_CANDIDATES = {   # feature -> domain (only those present are used)
    "Price": ["Monthly Charge", "Monthly Charges", "Total Charges", "Total Extra Data Charges", "Total Long Distance Charges",
              "Avg Monthly Long Distance Charges", "Total Refunds", "Offer"],
    "Support": ["Premium Tech Support", "Tech Support", "Online Security", "Device Protection Plan", "Device Protection"],
    "Offering": ["Internet Service", "Internet Type", "Avg Monthly GB Download", "Unlimited Data", "Streaming TV",
                 "Streaming Movies", "Streaming Music", "Multiple Lines", "Online Backup", "Phone Service"],
    "Relationship": ["Contract", "Tenure in Months", "Tenure Months", "Paperless Billing", "Payment Method",
                     "Referred a Friend", "Number of Referrals", "Total Revenue"],
    "Demographic": ["Gender", "Age", "Under 30", "Senior Citizen", "Married", "Partner", "Dependents",
                    "Number of Dependents", "Latitude", "Longitude", "Population"],
}
DMAP, seen = {}, set()
for d, names in DOMAIN_CANDIDATES.items():
    for n in names:
        c = col(n, required=False)
        if c and c not in seen: DMAP.setdefault(d, []).append(c); seen.add(c)
FEATS = [c for fs in DMAP.values() for c in fs]
print("\nFeature -> domain map used:"); [print(f"  {d:12s}: {fs}") for d, fs in DMAP.items()]

# ---------------------------------------------------------------------
# A. Leakage audit
# ---------------------------------------------------------------------
aud = []
for name in ["score", "satisfaction", "cltv"]:
    c = C[name]
    if c is not None:
        v = pd.to_numeric(raw[c], errors="coerce"); m = v.notna()
        aud.append({"Column": c, "Standalone AUC": round(roc_auc_score(y[m], v[m]), 3)})
aud.append({"Column": f"{C['reason']} (non-null)", "Standalone AUC": round(roc_auc_score(y, raw[C["reason"]].notna()), 3)})
if C["status"] is not None:
    aud.append({"Column": f"{C['status']} = Churned", "Standalone AUC": round(roc_auc_score(y, raw[C["status"]].astype(str).str.lower().eq("churned")), 3)})
aud = pd.DataFrame(aud).set_index("Column")
aud["Leakage (|AUC−0.5| > 0.35)"] = (aud["Standalone AUC"] - 0.5).abs() > 0.35
print("\n=== Study 2 leakage audit ==="); save(aud, "S2_T1_leakage")
SAT_OK = C["satisfaction"] is not None and not aud.loc[C["satisfaction"], "Leakage (|AUC−0.5| > 0.35)"]
if SAT_OK:
    DMAP["Satisfaction"] = [C["satisfaction"]]; FEATS.append(C["satisfaction"])
print("Satisfaction Score used as feature:", SAT_OK)

X = raw[FEATS].copy()
NUMF = [c for c in FEATS if pd.api.types.is_numeric_dtype(X[c]) or pd.to_numeric(X[c], errors="coerce").notna().mean() > 0.95]
for c in NUMF: X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)
CATF = [c for c in FEATS if c not in NUMF]
for c in CATF: X[c] = X[c].fillna("None").astype(str)
print(f"\n{len(FEATS)} features: {len(NUMF)} numeric, {len(CATF)} categorical")

# ---------------------------------------------------------------------
# B. Risk models, stratified 5-fold, out-of-fold probabilities and SHAP
# ---------------------------------------------------------------------
def pre(): return ColumnTransformer([("num", StandardScaler(), NUMF), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATF)])
def owners(p):
    o = []
    for n, t, cols in p.transformers_:
        if n == "cat": [o.extend([c] * len(cats)) for c, cats in zip(cols, t.categories_)]
        elif n == "num": o.extend(cols)
    return np.array(o)
MODELS = (["CatBoost"] if HAS_CB else []) + ["LogReg"]
SV = {m: np.zeros((len(X), len(FEATS))) for m in MODELS}; PROB = {m: np.zeros(len(X)) for m in MODELS}
skf = StratifiedKFold(K, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(skf.split(X, y)):
    for m in MODELS:
        p = pre().fit(X.iloc[tr]); Xtr, Xte = p.transform(X.iloc[tr]), p.transform(X.iloc[te])
        if m == "CatBoost":
            clf = CatBoostClassifier(iterations=500, depth=5, learning_rate=0.05, l2_leaf_reg=3, auto_class_weights="Balanced",
                                     random_seed=SEED + k, verbose=0, thread_count=-1).fit(Xtr, y[tr])
            sv = clf.get_feature_importance(Pool(Xte), type="ShapValues")[:, :-1]
        else:
            clf = LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000).fit(Xtr, y[tr])
            sv = clf.coef_[0] * (Xte - Xtr.mean(0))
        SV[m][te] = pd.DataFrame(sv, columns=owners(p)).T.groupby(level=0).sum().T[FEATS].values
        PROB[m][te] = clf.predict_proba(Xte)[:, 1]
perf = pd.DataFrame({m: {"AUC": roc_auc_score(y, PROB[m]), "PR-AUC": average_precision_score(y, PROB[m])} for m in MODELS}).T.round(4)
print("\n=== Study 2 risk models (out-of-fold, 5-fold) ==="); save(perf, "S2_T2_performance")
gi = pd.DataFrame({m: np.abs(SV[m]).mean(0) for m in MODELS}, index=FEATS); gi = gi.div(gi.sum(0), axis=1).sort_values(MODELS[0], ascending=False)
print("\nTop-12 global importance (share of |SHAP|):"); save(gi.head(12).round(4), "S2_T3_global_importance")

# ---------------------------------------------------------------------
# C. Domain-level reason validation (same protocol as Study 1)
# ---------------------------------------------------------------------
GROUPS = {"Competitor": ["Competitor offered higher download speeds", "Competitor offered more data", "Competitor made better offer", "Competitor had better devices"],
          "Price": ["Price too high", "Extra data charges", "Long distance charges", "Lack of affordable download/upload speed"],
          "Support": ["Attitude of support person", "Attitude of service provider", "Poor expertise of phone support", "Poor expertise of online support"],
          "Quality": ["Network reliability", "Product dissatisfaction", "Service dissatisfaction", "Lack of self-service on Website", "Limited range of services"]}
R2G = {r.lower(): g for g, rs in GROUPS.items() for r in rs}
G2D = {"Price": "Price", "Support": "Support", "Competitor": "Offering", "Quality": "Offering"}
reason = raw[C["reason"]].astype(str).str.strip().str.replace("’", "'")
rgroup = reason.str.lower().map(R2G)
val = np.where((y == 1) & rgroup.notna().values)[0]
td = rgroup.iloc[val].map(G2D).values
print(f"\nValidation churners with an actionable stated reason: {len(val)} | unmapped reasons among churners:",
      sorted(set(reason[(y == 1) & rgroup.isna().values]) - {"nan"}))
DOMS = ["Price", "Support", "Offering"]
def shares(att):
    pos = np.clip(att, 0, None); tot = pos.sum(1, keepdims=True); tot[tot == 0] = 1
    return pd.DataFrame({d: pos[:, [FEATS.index(f) for f in fs]].sum(1) / tot[:, 0] for d, fs in DMAP.items()})
rows = []
for m in MODELS:
    sh = shares(SV[m][val]); pred = np.array(DOMS)[sh[DOMS].values.argmax(1)]
    acc = (pred == td).mean(); perm = np.array([(pred == rng.permutation(td)).mean() for _ in range(N_PERM)])
    r = {"Explanation": f"SHAP-{m}", "Alignment acc.": round(acc, 3), "Permutation p": round((np.sum(perm >= acc) + 1) / (N_PERM + 1), 3),
         "Majority baseline": round(pd.Series(td).value_counts(normalize=True).iloc[0], 3), "Cohen κ": round(cohen_kappa_score(td, pred), 3)}
    for d in DOMS:
        lab = (td == d).astype(int); s = sh[d].values
        bs = [roc_auc_score(lab[b], s[b]) for b in (rng.integers(0, len(lab), len(lab)) for _ in range(N_BOOT))]
        r[f"AUC {d} [95% CI]"] = f"{roc_auc_score(lab, s):.3f} [{np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f}]"
    r["Non-actionable share"] = round(sh[[d for d in DMAP if d not in DOMS]].sum(1).mean(), 3)
    rows.append(r)
print("\n=== Study 2: explanations vs stated reasons (domain level) ==="); save(pd.DataFrame(rows).set_index("Explanation"), "S2_T4_reason_validation")

# ---------------------------------------------------------------------
# D. Feature-level test for reasons with a directly matching feature
# ---------------------------------------------------------------------
MATCH = {"Extra data charges": ["Total Extra Data Charges"],
         "Long distance charges": ["Total Long Distance Charges", "Avg Monthly Long Distance Charges"],
         "Competitor offered more data": ["Avg Monthly GB Download", "Unlimited Data"],
         "Competitor offered higher download speeds": ["Internet Type", "Avg Monthly GB Download"],
         "Lack of affordable download/upload speed": ["Internet Type", "Monthly Charge", "Monthly Charges"],
         "Price too high": ["Monthly Charge", "Monthly Charges"]}
fl, churn_idx = [], np.where(y == 1)[0]
for rsn, fnames in MATCH.items():
    grp = reason.iloc[churn_idx].str.lower().eq(rsn.lower()).values
    if grp.sum() < 10: continue
    for fn in fnames:
        f = col(fn, required=False)
        if f is None or f not in FEATS: continue
        j = FEATS.index(f)
        if f in NUMF:   # (i) does the feature itself differ for these churners?
            a, b = X[f].values[churn_idx][grp], X[f].values[churn_idx][~grp]
            diff = f"{a.mean():.2f} vs {b.mean():.2f} (MWU p={stats.mannwhitneyu(a, b).pvalue:.3g})"
        else:
            top = X[f].iloc[churn_idx][grp].value_counts(normalize=True).head(1)
            diff = f"{top.index[0]}: {100*top.iloc[0]:.0f}% vs {100*X[f].iloc[churn_idx][~grp].eq(top.index[0]).mean():.0f}%"
        for m in MODELS:   # (ii) does SHAP single out the matching feature for them?
            sv = SV[m][churn_idx]; rank = (np.argsort(np.argsort(-sv, 1), 1)[:, j] + 1)
            auc = roc_auc_score(grp.astype(int), np.clip(sv[:, j], 0, None))
            fl.append({"Stated reason": rsn, "n": int(grp.sum()), "Matching feature": f, "Feature value (reason vs other churners)": diff,
                       "Model": m, "SHAP AUC (feature, reason vs other churners)": round(auc, 3),
                       "Median SHAP rank (reason / others)": f"{np.median(rank[grp]):.0f} / {np.median(rank[~grp]):.0f}"})
print("\n=== Study 2: feature-level test ===")
save(pd.DataFrame(fl), "S2_T5_feature_level", index=False) if fl else print("No directly matching features found.")

# ---------------------------------------------------------------------
# E. Reason recoverability (why-model on churners) with / without geography
# ---------------------------------------------------------------------
yy = LabelEncoder().fit_transform(rgroup.iloc[val].values)
GEO = [c for c in [col("Latitude", required=False), col("Longitude", required=False), col("Population", required=False)] if c]
def why_auc(feats):
    num = [c for c in feats if c in NUMF]; cat = [c for c in feats if c in CATF]
    parts = ([("num", StandardScaler(), num)] if num else []) + ([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)] if cat else [])
    aucs = []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(val, yy):
        mdl = Pipeline([("p", ColumnTransformer(parts)), ("c", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                                                         class_weight="balanced", random_state=SEED))])
        mdl.fit(X.iloc[val[tr]][feats], yy[tr])
        aucs.append(roc_auc_score(yy[te], mdl.predict_proba(X.iloc[val[te]][feats]), multi_class="ovr", average="macro"))
    return f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}"
rec = pd.DataFrame([{"Feature set": "All features", "Macro AUC (4 groups)": why_auc(FEATS)},
                    {"Feature set": "Without geography", "Macro AUC (4 groups)": why_auc([f for f in FEATS if f not in GEO])},
                    {"Feature set": "Geography only", "Macro AUC (4 groups)": why_auc(GEO) if GEO else "—"}]).set_index("Feature set")
print("\n=== Study 2: reason recoverability ==="); save(rec, "S2_T6_recoverability")
json.dump({"path": path, "features": FEATS, "domains": DMAP, "satisfaction_used": bool(SAT_OK)}, open(os.path.join(OUT, "study2_config.json"), "w"), indent=1)
print("\nSaved to", OUT)

## Step 09: Extension 3: semi-synthetic study on four datasets


In [ ]:
# =====================================================================
# NB9 (single cell): MULTI-DATASET STUDY — when can attribution recover
#                    customers' reasons?  (semi-synthetic ground truth)
# Paper: Predicting Who Leaves Is Not Knowing Why: Validating Churn-Model
#        Explanations Against Customers' Stated Reasons
#
# Public churn datasets have no stated reasons, so we plant them. On each
# real dataset the churn model is trained on REAL churn; only the reason
# labels of churners are simulated, in two worlds:
#   World A "risk-aligned"   : reasons driven by the same features that drive churn risk
#   World B "risk-orthogonal": reasons driven by non-actionable features the risk model barely uses
#                              (mimics Study 1, where reasons follow geography)
# Reason ~ softmax(s * latent + log base-rate), base rates as in IBM (Offering 0.63).
# For each dataset x world x signal s: SHAP alignment vs majority, domain AUC,
# and whether the reasons are recoverable from features at all (why-model).
# Plus a Rashomon check: do equally accurate models agree on the top domain?
#
# Datasets (attach via + Add Input; missing ones are skipped):
#   IBM Telco      : NB1 output in /kaggle/working (telco_clean.csv + config.json)
#   Orange/BigML   : mnassrib/telecom-churn-datasets   (churn-bigml-80/20.csv)
#   Cell2Cell      : jpacse/datasets-for-churn-telecom (cell2celltrain.csv)
#   Iranian (UCI)  : royjafari/customer-churn          (Customer Churn.csv)
# RUNTIME: ~15-25 min on CPU.  Output: /kaggle/working/nb9/
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

SEED, N_REP, S_GRID, MAX_N = 42, 20, [0, 0.5, 1, 2, 4], 10000
BASE_RATE = {"Price": 0.15, "Support": 0.22, "Offering": 0.63}
RASHOMON_EPS = 0.02

import os, re, glob, json, itertools, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)
try:
    from catboost import CatBoostClassifier, Pool; HAS_CB = True
except ImportError:
    HAS_CB = False; print("catboost not found -> LogReg models only")

IN_W = os.environ.get("NB_IN", WORK_DIR)
OUT = os.path.join(os.environ.get("NB_OUT", WORK_DIR), "nb9")
FIG, TAB = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
for d in [OUT, FIG, TAB]: os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 600, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "pdf.fonttype": 42})
def save(df, name, index=True): df.to_csv(os.path.join(TAB, name + ".csv"), index=index); display(df)
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
DOMS = ["Price", "Support", "Offering"]

# ---------------------------------------------------------------------
# 0. Load datasets (detected by their columns, not by folder names)
# ---------------------------------------------------------------------
def read(p): return pd.read_excel(p) if p.lower().endswith((".xlsx", ".xls")) else pd.read_csv(p)
def header(p):
    try: return {norm(c) for c in (pd.read_excel(p, nrows=0) if p.lower().endswith((".xlsx", ".xls")) else pd.read_csv(p, nrows=0)).columns}
    except Exception: return set()
files = [p for p in glob.glob(os.environ.get("NB9_GLOB", DATA_DIR + "/**/*"), recursive=True) if p.lower().endswith((".csv", ".xlsx", ".xls"))]
HEAD = {p: header(p) for p in files}
SIGS = {"Orange/BigML": {"customerservicecalls", "totaldaycharge"},
        "Cell2Cell": {"monthlyrevenue", "droppedcalls"},
        "Iranian (UCI)": {"callfailure", "complains"}}
SLUGS = {"Orange/BigML": "mnassrib/telecom-churn-datasets",
         "Cell2Cell": "jpacse/datasets-for-churn-telecom",
         "Iranian (UCI)": "royjafari/customer-churn"}
# Not attached? download automatically (needs Settings -> Internet: On)
for name, sig in SIGS.items():
    if any(sig <= h for h in HEAD.values()): continue
    try:
        import kagglehub
        dl = kagglehub.dataset_download(SLUGS[name]); print(f"{name}: downloaded {SLUGS[name]}")
        for p in glob.glob(os.path.join(dl, "**/*"), recursive=True):
            if p.lower().endswith((".csv", ".xlsx", ".xls")): files.append(p); HEAD[p] = header(p)
    except Exception as e:
        print(f"{name}: auto-download failed ({repr(e)[:120]}) -> attach '{SLUGS[name]}' via + Add Input")
print(f"{len(files)} data files found:"); [print("  ", p) for p in files]
DATA = {}
try:
    cfg = json.load(open(os.path.join(IN_W, "config.json"))); d = pd.read_csv(os.path.join(IN_W, "telco_clean.csv"))
    feats = [f for f in cfg["num_features"] if f != "Avg Monthly Spend"] + cfg["bin_features"] + cfg["cat_features"]
    DATA["IBM Telco"] = (d[feats].copy(), d[cfg["target"]].astype(int).values)
except FileNotFoundError:
    print("IBM Telco skipped (run NB1 first)")
for name, sig in SIGS.items():
    hits = [p for p in files if sig <= HEAD[p] and "holdout" not in p.lower()]
    if not hits: print(f"{name} skipped: no attached file with columns {sig}"); continue
    df = pd.concat([read(p) for p in hits], ignore_index=True)
    df.columns = [c.strip() for c in df.columns]; df = df.drop_duplicates()
    tcol = next(c for c in df.columns if norm(c) in ("churn", "churnvalue", "churnlabel"))
    df = df[df[tcol].notna()]
    yv = df[tcol].astype(str).str.strip().str.lower().isin(["yes", "true", "1", "1.0"]).astype(int).values
    drop = [tcol] + [c for c in df.columns if norm(c) in ("customerid", "phonenumber", "id")]
    X = df.drop(columns=drop).reset_index(drop=True)
    if len(X) > MAX_N:   # stratified subsample for speed
        keep = np.sort(np.concatenate([rng.choice(np.where(yv == k)[0], int(round(MAX_N * (yv == k).mean())), replace=False) for k in (0, 1)]))
        X, yv = X.iloc[keep].reset_index(drop=True), yv[keep]
    DATA[name] = (X, yv)
    print(f"{name}: {len(hits)} file(s) -> {X.shape}")
assert DATA, "No dataset found."

# ---------------------------------------------------------------------
# Feature -> domain (same taxonomy as Study 1; rules listed in the paper's appendix)
# ---------------------------------------------------------------------
IBM_MAP = {**{f: "Price" for f in ["Monthly Charges", "Total Charges", "Charge Gap"]},
           **{f: "Support" for f in ["Tech Support", "Online Security", "Device Protection"]},
           **{f: "Offering" for f in ["Internet Service", "Streaming TV", "Streaming Movies", "Multiple Lines", "Online Backup", "Phone Service", "Num Services"]},
           **{f: "Relationship" for f in ["Contract", "Tenure Months", "Paperless Billing", "Payment Method"]}}
RULES = [("Relationship", r"paperless|payment|contract"),
         ("Support", r"customercare|customerservice|complain|retention|techsupport|support|security|deviceprotection"),
         ("Price", r"charge|revenue|price|amount|overage|tariff|refund"),
         ("Offering", r"minute|calls|called|failure|dropped|blocked|unanswered|usage|frequency|seconds|sms|gb|download|internet|streaming|plan|handset|equipment|roaming|vmail|voicemail|lines|backup|phoneservice|services|webcapable"),
         ("Relationship", r"tenure|length|monthsinservice|subs|referral|credit|status|customervalue|newcellphone")]
def domain_of(c):
    if c in IBM_MAP: return IBM_MAP[c]
    return next((d for d, rx in RULES if re.search(rx, norm(c))), "Other")

def prepare(X):
    X = X.copy(); num, cat = [], []
    for c in X.columns:
        v = pd.to_numeric(X[c], errors="coerce")
        if v.notna().mean() > 0.95 and X[c].nunique() > 2 or pd.api.types.is_numeric_dtype(X[c]):
            X[c] = v.fillna(v.median()); num.append(c)
        elif X[c].nunique() <= 50:
            X[c] = X[c].astype(str).fillna("NA"); cat.append(c)
    return X[num + cat], num, cat

# ---------------------------------------------------------------------
# 1. Risk models (5-fold OOF) + SHAP per original feature
# ---------------------------------------------------------------------
SPECS = ([("CatBoost-d4", "cb", 4), ("CatBoost-d6", "cb", 6), ("CatBoost-d8", "cb", 8)] if HAS_CB else []) + \
        [("LogReg-C1", "lr", 1.0), ("LogReg-C0.1", "lr", 0.1)]
PRIMARY = ["CatBoost-d6", "LogReg-C1"] if HAS_CB else ["LogReg-C1"]
def owners(p):
    o = []
    for n, t, cols in p.transformers_:
        if n == "cat" and len(cols) and hasattr(t, "categories_"): [o.extend([c] * len(cats)) for c, cats in zip(cols, t.categories_)]
        elif n == "num": o.extend(cols)
    return np.array(o)
def fit_all(X, y, num, cat):
    SV = {m: np.zeros((len(X), X.shape[1])) for m, _, _ in SPECS}; PR = {m: np.zeros(len(X)) for m, _, _ in SPECS}
    for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)):
        pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)]).fit(X.iloc[tr])
        Xtr, Xte, own = pre.transform(X.iloc[tr]), pre.transform(X.iloc[te]), owners(pre)
        for m, kind, h in SPECS:
            if kind == "cb":
                clf = CatBoostClassifier(iterations=400, depth=h, learning_rate=0.05, auto_class_weights="Balanced",
                                         random_seed=SEED + k + h, verbose=0, thread_count=-1).fit(Xtr, y[tr])
                sv = clf.get_feature_importance(Pool(Xte), type="ShapValues")[:, :-1]
            else:
                clf = LogisticRegression(C=h, class_weight="balanced", max_iter=5000).fit(Xtr, y[tr])
                sv = clf.coef_[0] * (Xte - Xtr.mean(0))
            SV[m][te] = pd.DataFrame(sv, columns=own).T.groupby(level=0).sum().T[list(X.columns)].values
            PR[m][te] = clf.predict_proba(Xte)[:, 1]
    return SV, PR

def domain_shares(sv, dom):
    pos = np.clip(sv, 0, None); tot = pos.sum(1, keepdims=True); tot[tot == 0] = 1
    return {d: pos[:, dom == d].sum(1) / tot[:, 0] for d in ["Price", "Support", "Offering", "Relationship", "Other"]}

# ---------------------------------------------------------------------
# 2. Latent reason drivers for the two worlds
# ---------------------------------------------------------------------
def latents(X, y, num, cat, dom_cols, gi_feat):
    pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)]).fit(X)
    Z, own = pre.transform(X), owners(pre)
    Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-9)
    sgn = np.sign([np.corrcoef(Z[:, j], y)[0, 1] if Z[:, j].std() > 0 else 0 for j in range(Z.shape[1])])
    dom_own = np.array([dom_cols[o] for o in own])
    A = np.column_stack([(Z[:, dom_own == d] * sgn[dom_own == d]).mean(1) if (dom_own == d).any() else rng.normal(size=len(Z)) for d in DOMS])
    # World B: random projections of NON-ACTIONABLE features (Relationship/Other) that the risk model
    # uses least (bottom half of their global |SHAP|) -> reasons live outside the attribution domains,
    # as geography does in Study 1
    na = gi_feat[[dom_cols[c] not in DOMS for c in gi_feat.index]]
    low = set(na.sort_values().index[: max(1, len(na) // 2)])
    Zl = Z[:, np.isin(own, list(low))]
    B = Zl @ rng.normal(size=(Zl.shape[1], 3)) if Zl.shape[1] else rng.normal(size=(len(Z), 3))
    st = lambda M: (M - M.mean(0)) / (M.std(0) + 1e-9)
    return st(A), st(B), Z, sorted(low)
def draw(L, s):
    logit = s * L + np.log([BASE_RATE[d] for d in DOMS])
    P = np.exp(logit - logit.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    u = rng.random(len(P))[:, None]
    return np.array(DOMS)[(u > P.cumsum(1)).sum(1).clip(0, 2)]
def why_auc(Z, lab):
    yy = pd.Series(lab).map({d: i for i, d in enumerate(DOMS)}).values; aucs = []
    if np.bincount(yy, minlength=3).min() < 5: return np.nan
    for tr, te in StratifiedKFold(3, shuffle=True, random_state=SEED).split(Z, yy):
        m = HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=0.05, random_state=SEED).fit(Z[tr], yy[tr])
        aucs.append(roc_auc_score(yy[te], m.predict_proba(Z[te]), multi_class="ovr", average="macro"))
    return np.mean(aucs)

# ---------------------------------------------------------------------
# 3. Run
# ---------------------------------------------------------------------
summ, synth, rash = [], [], []
t0 = time.time()
for name, (X0, y) in DATA.items():
    X, num, cat = prepare(X0)
    dom_cols = {c: domain_of(c) for c in X.columns}; dom = np.array([dom_cols[c] for c in X.columns])
    SV, PR = fit_all(X, y, num, cat)
    aucs = {m: roc_auc_score(y, PR[m]) for m in SV}
    gi = pd.Series(np.abs(SV[PRIMARY[0]]).mean(0), index=X.columns)
    gshare = gi.groupby(dom).sum() / gi.sum()
    summ.append({"Dataset": name, "n": len(X), "Churn rate": round(y.mean(), 3), "Features": X.shape[1],
                 **{f"# {d}": int((dom == d).sum()) for d in DOMS + ["Relationship", "Other"]},
                 **{f"AUC {m}": round(aucs[m], 3) for m in PRIMARY},
                 **{f"|SHAP| share {d}": round(gshare.get(d, 0), 3) for d in DOMS}})
    ch = np.where(y == 1)[0]
    A, B, Z, low = latents(X, y, num, cat, dom_cols, gi)
    A, B, Zc = A[ch], B[ch], Z[ch]
    shares = {m: domain_shares(SV[m][ch], dom) for m in PRIMARY}
    ind = {m: np.array(DOMS)[np.column_stack([shares[m][d] for d in DOMS]).argmax(1)] for m in PRIMARY}
    for world, L in [("A risk-aligned", A), ("B risk-orthogonal", B)]:
        for s in S_GRID:
            rec = {m: {"gain": [], "auc": []} for m in PRIMARY}; why = []
            for r in range(N_REP):
                lab = draw(L, s); maj = pd.Series(lab).value_counts(normalize=True).iloc[0]
                for m in PRIMARY:
                    rec[m]["gain"].append((ind[m] == lab).mean() - maj)
                    rec[m]["auc"].append(np.nanmean([roc_auc_score(lab == d, shares[m][d]) for d in DOMS if 0 < (lab == d).mean() < 1]))
                if s == 2 and r < 3: why.append(why_auc(Zc, lab))
            for m in PRIMARY:
                synth.append({"Dataset": name, "World": world, "s": s, "Model": m,
                              "Accuracy gain vs majority": np.mean(rec[m]["gain"]), "Gain SD": np.std(rec[m]["gain"]),
                              "Mean domain AUC": np.mean(rec[m]["auc"]), "AUC SD": np.std(rec[m]["auc"]),
                              "Why-model AUC (s=2)": np.nanmean(why) if why else np.nan})
    # Rashomon: models within EPS of the best AUC; agreement on churners' top actionable domain
    good = [m for m in SV if aucs[m] >= max(aucs.values()) - RASHOMON_EPS]
    tops = {m: np.array(DOMS)[np.column_stack([domain_shares(SV[m][ch], dom)[d] for d in DOMS]).argmax(1)] for m in good}
    pair = [(tops[a] == tops[b]).mean() for a, b in itertools.combinations(good, 2)]
    allag = np.mean([len({tops[m][i] for m in good}) == 1 for i in range(len(ch))]) if len(good) > 1 else np.nan
    rash.append({"Dataset": name, "Models in Rashomon set (ε=0.02)": ", ".join(good),
                 "AUC range": f"{min(aucs[m] for m in good):.3f}–{max(aucs[m] for m in good):.3f}",
                 "Mean pairwise top-domain agreement (%)": round(100 * np.mean(pair), 1) if pair else np.nan,
                 "All models agree (%)": round(100 * allag, 1) if pair else np.nan})
    print(f"{name}: done | {(time.time()-t0)/60:.1f} min | World-B drivers: {low[:6]}{' ...' if len(low) > 6 else ''}")

print("\n=== Datasets and risk models ==="); save(pd.DataFrame(summ).set_index("Dataset"), "T26_datasets")
syn = pd.DataFrame(synth)
syn.round(3).to_csv(os.path.join(TAB, "T27_semisynthetic_full.csv"), index=False)
key = syn[syn.s.isin([0, 2])].copy()
key["Gain [SD]"] = key.apply(lambda r: f"{r['Accuracy gain vs majority']:+.3f} [{r['Gain SD']:.3f}]", axis=1)
key["Domain AUC [SD]"] = key.apply(lambda r: f"{r['Mean domain AUC']:.3f} [{r['AUC SD']:.3f}]", axis=1)
key = key.pivot_table(index=["Dataset", "World", "Model"], columns="s", values=["Gain [SD]", "Domain AUC [SD]"], aggfunc="first")
key.columns = [f"{a} (s={b})" for a, b in key.columns]
key = key.join(syn[syn.s == 2].groupby(["Dataset", "World", "Model"])["Why-model AUC (s=2)"].first().round(3))
print("\n=== Semi-synthetic reason recovery (s=0: no signal, s=2: strong signal) ==="); save(key, "T28_semisynthetic_key")
print("\n=== Rashomon agreement on churners' top actionable domain ==="); save(pd.DataFrame(rash).set_index("Dataset"), "T29_rashomon")

# ---------------------------------------------------------------------
# Figure 10: domain AUC vs reason signal strength, per dataset
# ---------------------------------------------------------------------
dsn = list(DATA); fig, axs = plt.subplots(1, len(dsn), figsize=(2.6 * len(dsn) + 0.6, 2.8), sharey=True, squeeze=False)
col = {"A risk-aligned": "#3B5B92", "B risk-orthogonal": "#C0392B"}
for ax, dn in zip(axs[0], dsn):
    for (w, m), g in syn[syn.Dataset == dn].groupby(["World", "Model"]):
        ls = "-" if m == PRIMARY[0] else "--"
        ax.errorbar(g.s, g["Mean domain AUC"], yerr=g["AUC SD"], color=col[w], ls=ls, marker="o", ms=3, capsize=2,
                    label=f"{w.split()[0]}: {w.split(' ', 1)[1]} · {m}")
    ax.axhline(0.5, color="grey", lw=0.8, ls=":"); ax.set_title(dn); ax.set_xlabel("Reason signal strength s")
axs[0][0].set_ylabel("Mean domain AUC (SHAP share)")
h, l = axs[0][0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0), fontsize=8)
plt.tight_layout()
for ext in ["png", "pdf"]: plt.savefig(os.path.join(FIG, f"Fig_10_semisynthetic.{ext}"), bbox_inches="tight")
plt.show()
print("\nSaved to", OUT, "->", sorted(os.listdir(TAB)))

## Step 10: Reliability of stated reasons across releases


In [ ]:
# =====================================================================
# NB10 (single cell): How reliable are the benchmark's stated reasons?
#   A. Same customers, two releases: do churn labels, features and stated
#      reasons agree between the 33-variable sample (Study 1) and the
#      11.1.3+ release (Study 2)?
#   B. Are reasons clustered by location? Share of churner pairs in the same
#      ZIP code / city with the same reason group vs a permutation null.
# RUNTIME: < 1 min.  Output: /kaggle/working/nb10/
# =====================================================================
import os, glob
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")  # outputs; later steps read earlier outputs here
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/input")    # raw datasets (Kaggle input or a local folder)
try:
    display
except NameError:  # running as a plain script outside Jupyter
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)

SEED, N_PERM = 42, 2000
import os, re, glob, numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
rng = np.random.default_rng(SEED)
OUT = os.path.join(WORK_DIR, "nb10"); os.makedirs(OUT, exist_ok=True)
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
files = glob.glob(DATA_DIR + "/**/*.xlsx", recursive=True) + glob.glob(DATA_DIR + "/**/*.csv", recursive=True)
def load(pred):
    for p in files:
        d = pd.read_excel(p) if p.endswith(".xlsx") else pd.read_csv(p)
        if pred({norm(c) for c in d.columns}): return p, d
    raise FileNotFoundError
p1, A = load(lambda c: "churnreason" in c and "totalextradatacharges" not in c)
p2, B = load(lambda c: "totalextradatacharges" in c and "churnreason" in c)
print("Release 1:", p1, A.shape, "\nRelease 2:", p2, B.shape)
def col(d, *names):
    m = {norm(c): c for c in d.columns}
    return next(m[norm(n)] for n in names if norm(n) in m)
GROUPS = {"Competitor": ["Competitor offered higher download speeds", "Competitor offered more data", "Competitor made better offer", "Competitor had better devices"],
          "Price": ["Price too high", "Extra data charges", "Long distance charges", "Lack of affordable download/upload speed"],
          "Support": ["Attitude of support person", "Attitude of service provider", "Poor expertise of phone support", "Poor expertise of online support"],
          "Quality": ["Network reliability", "Product dissatisfaction", "Service dissatisfaction", "Lack of self-service on Website", "Limited range of services"],
          "Uncontrollable": ["Moved", "Deceased"], "Unknown": ["Don't know"]}
R2G = {r.lower(): g for g, rs in GROUPS.items() for r in rs}
grp = lambda s: s.astype(str).str.strip().str.replace("’", "'").str.lower().map(R2G)

a = pd.DataFrame({"id": A[col(A, "CustomerID", "Customer ID")], "churn1": A[col(A, "Churn Label")].astype(str).str.lower().eq("yes"),
                  "reason1": A[col(A, "Churn Reason")], "tenure1": A[col(A, "Tenure Months")], "mc1": A[col(A, "Monthly Charges")],
                  "zip1": A[col(A, "Zip Code")], "city1": A[col(A, "City")]})
b = pd.DataFrame({"id": B[col(B, "Customer ID", "CustomerID")], "churn2": B[col(B, "Churn Label")].astype(str).str.lower().eq("yes"),
                  "reason2": B[col(B, "Churn Reason")], "tenure2": B[col(B, "Tenure in Months", "Tenure Months")],
                  "mc2": B[col(B, "Monthly Charge", "Monthly Charges")]})
m = a.merge(b, on="id"); print(f"\nMatched customers: {len(m)}")

# ---------- A. agreement between releases
both = m[m.churn1 & m.churn2].copy()
both["g1"], both["g2"] = grp(both.reason1), grp(both.reason2)
same_reason = (both.reason1.astype(str).str.strip().str.lower() == both.reason2.astype(str).str.strip().str.lower())
# chance agreement if reasons were assigned independently with the observed marginals
p1r = both.reason1.astype(str).str.lower().value_counts(normalize=True); p2r = both.reason2.astype(str).str.lower().value_counts(normalize=True)
p1g = both.g1.value_counts(normalize=True); p2g = both.g2.value_counts(normalize=True)
rowsA = {"Customers matched by ID": len(m),
         "Churn label identical (%)": 100 * (m.churn1 == m.churn2).mean(),
         "Tenure identical (%)": 100 * (m.tenure1 == m.tenure2).mean(),
         "Monthly charge identical (±0.01) (%)": 100 * (np.abs(m.mc1 - m.mc2) <= 0.01).mean(),
         "Churners in both releases": len(both),
         "Same stated reason (%)": 100 * same_reason.mean(),
         "  expected if independent (%)": 100 * (p1r * p2r.reindex(p1r.index).fillna(0)).sum(),
         "Same reason group (%)": 100 * (both.g1 == both.g2).mean(),
         "  expected if independent (%) ": 100 * (p1g * p2g.reindex(p1g.index).fillna(0)).sum(),
         "Cohen's κ, reason group": cohen_kappa_score(both.g1.fillna("NA"), both.g2.fillna("NA"))}
TA = pd.Series(rowsA).round(3).to_frame("Value"); TA.to_csv(f"{OUT}/T30_release_agreement.csv")
print("\n=== A. Same customers, two releases ==="); display(TA)
ct = pd.crosstab(both.g1.rename("Release 1 (Study 1)"), both.g2.rename("Release 2 (Study 2)"))
ct.to_csv(f"{OUT}/T31_reason_group_crosstab.csv"); display(ct)

# ---------- B. geographic clustering of reasons (Study 1 churners)
ch = a[a.churn1].copy(); ch["g"] = grp(ch.reason1)
def same_pair_share(key, labels):
    df = pd.DataFrame({"k": key.values, "g": labels})
    s, n = 0, 0
    for _, g in df.groupby("k"):
        c = g["g"].value_counts().values; t = len(g)
        if t < 2: continue
        s += (c * (c - 1)).sum() / 2; n += t * (t - 1) / 2
    return s / n if n else np.nan
rowsB = []
for name, key in [("ZIP code", ch.zip1), ("City", ch.city1)]:
    obs = same_pair_share(key, ch.g.values)
    null = np.array([same_pair_share(key, rng.permutation(ch.g.values)) for _ in range(N_PERM // 10)])
    rowsB.append({"Grouping": name, "Groups with ≥2 churners": int((key.value_counts() >= 2).sum()),
                  "Same-group pairs, observed (%)": round(100 * obs, 1), "Permutation mean (%)": round(100 * null.mean(), 1),
                  "Permutation p": round((np.sum(null >= obs) + 1) / (len(null) + 1), 4)})
TB = pd.DataFrame(rowsB).set_index("Grouping"); TB.to_csv(f"{OUT}/T32_geo_clustering.csv")
print("\n=== B. Do churners in the same location state the same reason group? ==="); display(TB)
print("Saved to", OUT)